# FT03 — Data Discovery: USGS

## Objetivo
Explorar a API pública USGS Water Data API para identificar
endpoints, parâmetros, estruturas JSON e possibilidades
de utilização no pipeline ETL.

## Ambiente
- Python 3.13.0
- Pandas 3.1.1
- Requests 2.34.2
- Jupyter Notebook

## Etapas
1. Identificação dos endpoints.
2. Primeira requisição HTTP.
3. Análise do status HTTP.
4. Inspeção da resposta JSON.
5. Identificação de estruturas aninhadas.
6. Avaliação dos dados para transformação.

## 1. Importação de bibliotecas e validação do ambiente

### Objetivo
Importar as bibliotecas Python necessárias para a investigação da API USGS e verificar as versões do interpretador e das principais dependências utilizadas no projeto.

### Justificativa
Antes de iniciar a extração e manipulação dos dados, precisamos garantir que o ambiente de execução esteja corretamente configurado.

A validação permite identificar:
- Qual versão do Python está sendo utilizada.
- Qual interpretador está executando o notebook.
- Quais versões das bibliotecas estão instaladas.
- Se as dependências necessárias estão disponíveis.

### Bibliotecas utilizadas

| Biblioteca | Finalidade |
|---|---|
| `sys` | Acessar informações do ambiente Python e do interpretador. |
| `pandas` | Manipulação, transformação e análise de dados tabulares. |
| `requests` | Realizar requisições HTTP para consumir APIs. |
| `json` | Interpretar e manipular estruturas de dados JSON. |
| `re` | Trabalhar com expressões regulares para pesquisa textual. |

### Explicação técnica

- `import`: carrega uma biblioteca ou módulo para utilização no código.
- `as pd`: cria um apelido para a biblioteca Pandas, seguindo a convenção amplamente utilizada na comunidade Python.
- `sys.version`: retorna informações sobre a versão do Python.
- `sys.executable`: informa o caminho do executável Python utilizado pelo kernel.
- `pd.__version__`: retorna a versão instalada do Pandas.
- `requests.__version__`: retorna a versão instalada do Requests.
- `print()`: apresenta as informações no console de saída do notebook.

### Resultado da execução

- Python: 3.13.0
- Pandas: 3.1.1
- Requests: 2.34.2
- Interpretador: ambiente virtual `.venv` do projeto FT03-Python.

### Conclusão

O ambiente de execução foi identificado e as bibliotecas principais foram carregadas com sucesso. O notebook está preparado para iniciar as requisições à API USGS e realizar a exploração dos dados.

In [2]:
import sys
import pandas as pd
import requests
import json
import re

print("Python:", sys.version)
print("Interpretador:", sys.executable)
print("Pandas:", pd.__version__)
print("Requests:", requests.__version__)

Python: 3.13.0 (tags/v3.13.0:60403a5, Oct  7 2024, 09:38:07) [MSC v.1941 64 bit (AMD64)]
Interpretador: c:\Users\mayal\OneDrive\Documentos\01. Office\01. Acadêmico\Engenharia de Dados\FT03-Python\.venv\Scripts\python.exe
Pandas: 3.1.1
Requests: 2.34.2


## 2. Configuração e primeira requisição à API USGS

### Objetivo

Configurar o acesso à API pública USGS Water Data API e realizar a primeira requisição HTTP para consultar o catálogo de características ambientais disponíveis.

### Justificativa

Precisamos estabelecer uma comunicação entre nosso ambiente Python e a fonte externa de dados.

Essa etapa permite verificar:
- Se o endpoint da API está acessível.
- Se a requisição HTTP foi processada com sucesso.
- Qual URL foi efetivamente consultada.
- Qual formato de conteúdo foi retornado pelo serviço.

### Conceitos técnicos

**API (Application Programming Interface):** interface que permite a comunicação entre diferentes sistemas por meio de regras e estruturas previamente definidas.

**HTTP (Hypertext Transfer Protocol):** protocolo utilizado para comunicação entre clientes e servidores na web.

**Endpoint:** endereço específico de uma API responsável por disponibilizar determinado recurso ou funcionalidade.

**GET:** método HTTP utilizado para solicitar informações de um servidor.

**Parâmetros de consulta:** informações adicionais enviadas na URL para definir características da solicitação.

### Configurações utilizadas

| Variável | Finalidade |
|---|---|
| `BASE_URL` | Define o endereço-base da API USGS. |
| `ENDPOINT` | Define o recurso específico que será consultado. |
| `PARAMS` | Armazena os parâmetros enviados na requisição. |
| `response` | Armazena a resposta HTTP recebida do servidor. |

### Explicação dos principais comandos

- `f-string`: permite inserir variáveis diretamente em uma string.
- `requests.get()`: realiza uma requisição HTTP utilizando o método GET.
- `params=PARAMS`: envia os parâmetros de consulta à API.
- `timeout=30`: estabelece um limite de 30 segundos para a espera pela resposta.
- `response.status_code`: retorna o código HTTP recebido.
- `response.url`: apresenta a URL efetivamente consultada.
- `response.headers.get("Content-Type")`: identifica o tipo de conteúdo informado pelo servidor.


In [3]:
# Configuração da API USGS

BASE_URL = "https://api.waterdata.usgs.gov/samples-data"

ENDPOINT = f"{BASE_URL}/codeservice/characteristics"

PARAMS = {
    "mimeType": "application/json"
}

# Requisição HTTP
response = requests.get(
    ENDPOINT,
    params=PARAMS,
    timeout=30
)

# Resultado da conexão
print("Status HTTP:", response.status_code)
print("URL consultada:", response.url)
print("Content-Type:", response.headers.get("Content-Type"))

Status HTTP: 200
URL consultada: https://api.waterdata.usgs.gov/samples-data/codeservice/characteristics?mimeType=application%2Fjson
Content-Type: application/json


### Resultado da execução

| Indicador | Resultado |
|---|---|
| Status HTTP | 200 |
| Content-Type | application/json |
| Endpoint | codeservice/characteristics |
| Tempo de execução observado | 2,8 segundos |

### Interpretação

O código HTTP 200 indica que a requisição foi processada com sucesso pelo servidor.

O cabeçalho `Content-Type: application/json` informa que a resposta está sendo disponibilizada no formato JSON.

A URL apresentada confirma o envio do parâmetro `mimeType=application/json`.

### Conclusão

A comunicação entre o ambiente Python e a API USGS foi estabelecida com sucesso.

A resposta recebida confirma que podemos avançar para a inspeção do conteúdo JSON e investigar a estrutura dos dados disponibilizados pela API.

## 3. Conversão e inspeção da resposta JSON

### Objetivo

Converter a resposta HTTP recebida da API USGS em um objeto Python, identificar sua estrutura hierárquica e realizar uma inspeção inicial dos registros retornados.

### Justificativa

A resposta de uma API geralmente é disponibilizada em formatos estruturados, como JSON.

Embora esse formato seja adequado para comunicação entre sistemas, precisamos convertê-lo em estruturas Python para acessar seus elementos, investigar os campos disponíveis e preparar os dados para posterior transformação em DataFrame.

### Conceitos técnicos

**JSON (JavaScript Object Notation):** formato textual utilizado para representar dados estruturados, organizados em objetos, listas, pares de chave e valor.

**Dicionário Python (`dict`):** estrutura de dados que armazena informações em pares de chave e valor.

**Lista Python (`list`):** estrutura que permite armazenar uma sequência de elementos.

**Estrutura hierárquica:** organização de dados em diferentes níveis, permitindo que objetos e listas sejam agrupados dentro de outras estruturas.

### Explicação dos comandos

- `response.json()`: converte o conteúdo JSON da resposta HTTP em estruturas de dados Python.
- `type(dados)`: identifica o tipo de objeto retornado.
- `dados.keys()`: apresenta as chaves existentes no dicionário principal.
- `len(dados["data"])`: contabiliza os registros armazenados na chave `data`.
- `dados["data"][:2]`: seleciona os dois primeiros registros da lista.
- `json.dumps()`: serializa um objeto Python para uma representação textual JSON.
- `indent=2`: organiza a saída com indentação de dois espaços, facilitando a leitura.
- `ensure_ascii=False`: preserva caracteres não ASCII na representação textual.

### Estrutura identificada

A resposta apresenta duas chaves principais:

- `data`: contém os registros das características ambientais.
- `count`: contém a informação de quantidade total disponibilizada pela API.

Cada registro possui campos como:

- `characteristicGroup`: grupo da característica.
- `parameterCode`: código associado ao parâmetro.
- `characteristicName`: nome da característica.
- `characteristicNameUserSupplied`: descrição detalhada fornecida para a característica.

### Resultado da execução

- Tipo de retorno: `dict`.
- Chaves principais: `data` e `count`.
- Quantidade de registros em `data`: 11.946.
- Registros inspecionados: 2.

### Conclusão

A resposta da API foi convertida com sucesso em um dicionário Python.

A inspeção permitiu identificar a estrutura hierárquica do JSON, conhecer os campos disponíveis e confirmar que a chave `data` contém os registros que serão utilizados nas próximas etapas da exploração.

A inspeção de uma amostra dos registros permite compreender a estrutura dos dados antes de realizar sua transformação para um formato tabular.

In [4]:
# Converter a resposta HTTP para objeto Python
dados = response.json()

# Inspecionar a estrutura
print("Tipo de retorno:", type(dados))
print("Chaves principais:", dados.keys())

# Quantidade de registros
print("Quantidade de registros:", len(dados["data"]))

# Visualizar os dois primeiros registros
print(json.dumps(dados["data"][:2], indent=2, ensure_ascii=False))

Tipo de retorno: <class 'dict'>
Chaves principais: dict_keys(['data', 'count'])
Quantidade de registros: 11946
[
  {
    "characteristicGroup": "Organics, Other",
    "parameterCode": "67995",
    "characteristicName": "Hydroxy-amitriptyline, 10-",
    "characteristicNameUserSupplied": "10-Hydroxy-amitriptyline, water, filtered, recoverable"
  },
  {
    "characteristicGroup": "Organics, Other",
    "parameterCode": "52417",
    "characteristicName": "1,1,1,2-Tetrachloroethane",
    "characteristicNameUserSupplied": "1,1,1,2-Tetrachloroethane, bed sediment (dry mass basis), recoverable"
  }
]


### 3.1. Resultado e análise da resposta JSON

A execução apresentou os seguintes resultados:

| Indicador | Resultado |
|---|---|
| Tipo do objeto | `dict` |
| Chaves principais | `data`, `count` |
| Registros na lista `data` | 11.946 |
| Registros visualizados | 2 |

#### Análise da estrutura

A resposta apresenta uma estrutura hierárquica:

- O objeto principal é um dicionário Python.
- A chave `data` contém uma lista de registros.
- Cada elemento dessa lista representa uma característica ambiental.
- Cada característica possui atributos identificadores e descritivos.

#### Observação técnica

A quantidade de 11.946 foi obtida por meio de `len(dados["data"])`, ou seja, representa a quantidade de elementos presentes na lista retornada.

Embora a resposta também apresente a chave `count`, seu conteúdo ainda não foi inspecionado individualmente.

#### Conclusão

A estrutura JSON foi interpretada com sucesso e os registros estão disponíveis em estruturas nativas do Python.

Essa etapa estabelece a base para transformar os dados semiestruturados em uma estrutura tabular utilizando Pandas.

## 4. Conversão dos registros JSON para DataFrame

### Objetivo

Converter a lista de registros retornada pela API USGS em um DataFrame Pandas, permitindo visualizar os dados em formato tabular e investigar suas dimensões, colunas e tipos de dados.

### Justificativa

Após identificar a estrutura hierárquica da resposta JSON, precisamos organizar os registros em uma estrutura adequada para análise e transformação.

O Pandas disponibiliza o objeto DataFrame, que permite trabalhar com dados organizados em linhas e colunas, facilitando operações de filtragem, agrupamento, tratamento e análise.

### Conceitos técnicos

**DataFrame:** estrutura bidimensional do Pandas, organizada em linhas e colunas.

**Shape:** propriedade que retorna as dimensões do DataFrame, representadas por uma tupla (linhas, colunas).

**Columns:** propriedade que permite identificar os nomes das colunas existentes.

**Dtypes:** propriedade que apresenta os tipos de dados de cada coluna.

**Head:** método utilizado para visualizar os primeiros registros de um DataFrame.

### Explicação dos comandos

- `pd.DataFrame(dados["data"])`: converte a lista de registros JSON em um DataFrame.
- `.shape`: retorna a quantidade de linhas e colunas.
- `.columns`: retorna os nomes das colunas.
- `.tolist()`: converte os nomes das colunas em uma lista Python.
- `.dtypes`: identifica os tipos de dados de cada coluna.
- `.head()`: apresenta os cinco primeiros registros.
- `display()`: apresenta o objeto em formato visual no ambiente Jupyter Notebook.

### Resultado esperado

Um DataFrame contendo os registros disponibilizados pela API, acompanhado de informações sobre dimensões, colunas e tipos de dados.

In [5]:
# Converter a lista de registros em DataFrame
df_caracteristicas = pd.DataFrame(dados["data"])

# Inspecionar dimensões
print("Dimensões:", df_caracteristicas.shape)

# Identificar colunas
print("\nColunas:")
print(df_caracteristicas.columns.tolist())

# Identificar tipos de dados
print("\nTipos de dados:")
print(df_caracteristicas.dtypes)

# Visualizar registros
display(df_caracteristicas.head())

Dimensões: (11946, 4)

Colunas:
['characteristicGroup', 'parameterCode', 'characteristicName', 'characteristicNameUserSupplied']

Tipos de dados:
characteristicGroup               str
parameterCode                     str
characteristicName                str
characteristicNameUserSupplied    str
dtype: object


,characteristicGroup,parameterCode,characteristicName,characteristicNameUserSupplied
0,"Organics, Other",67995,"Hydroxy-amitriptyline, 10-","10-Hydroxy-amitriptyline, water, filtered, rec..."
1,"Organics, Other",52417,"1,1,1,2-Tetrachloroethane","1,1,1,2-Tetrachloroethane, bed sediment (dry m..."
2,"Organics, Other",62235,"1,1,1,2-Tetrachloroethane","1,1,1,2-Tetrachloroethane, soil (dry mass basi..."
3,"Organics, Other",30089,"1,1,1,2-Tetrachloroethane","1,1,1,2-Tetrachloroethane, soil (dry mass basi..."
4,"Organics, Other",77562,"1,1,1,2-Tetrachloroethane","1,1,1,2-Tetrachloroethane, water, unfiltered, ..."


### 4.1. Resultado e análise estrutural

A conversão dos registros JSON para DataFrame foi realizada com sucesso.

**Dimensões identificadas:** 11.946 linhas e 4 colunas.

**Colunas identificadas:**
- `characteristicGroup`
- `parameterCode`
- `characteristicName`
- `characteristicNameUserSupplied`

**Tipos de dados:** todas as quatro colunas foram identificadas como texto (`str`).

#### Análise técnica

A estrutura tabular permite trabalhar individualmente com cada atributo dos registros.

A identificação dos tipos de dados também é importante para evitar conversões inadequadas durante as próximas etapas de transformação.

O campo `parameterCode` foi mantido como texto, preservando sua representação como código identificador.

#### Conclusão

O DataFrame `df_caracteristicas` foi criado com sucesso e está disponível para as próximas etapas de exploração, filtragem e análise das características ambientais disponibilizadas pela API USGS.

## 5. Agrupamento e contagem de características por grupo

### Objetivo

Identificar a quantidade de características ambientais existentes em cada grupo, utilizando operações de agrupamento e agregação do Pandas.

### Justificativa

Após transformar os registros JSON em um DataFrame, podemos começar a explorar a distribuição das características ambientais.

O agrupamento permite organizar os registros de acordo com valores comuns de uma coluna e calcular indicadores para cada grupo.

Neste caso, queremos identificar quais grupos possuem a maior quantidade de características cadastradas na API USGS.

### Conceitos técnicos

**Group By:** operação que agrupa registros de acordo com uma ou mais colunas.

**Size:** método que contabiliza a quantidade de registros existentes em cada grupo, incluindo linhas com outros campos nulos.

**Sort Values:** método utilizado para ordenar os resultados de acordo com os valores de uma coluna ou série.

**Ascending:** parâmetro que define a ordem da classificação. Quando configurado como `False`, apresenta os maiores valores primeiro.

### Explicação dos comandos

- `df_caracteristicas.groupby("characteristicGroup")`: agrupa os registros pela coluna de grupo da característica.
- `.size()`: contabiliza a quantidade de registros em cada grupo.
- `.sort_values(ascending=False)`: ordena os grupos da maior para a menor quantidade.
- `display(grupos.head(15))`: apresenta os 15 primeiros resultados.

### Resultado esperado

Uma relação dos grupos de características ambientais, ordenada pela quantidade de registros existentes em cada grupo.

In [6]:
# Quantidade de características por grupo
grupos = (
    df_caracteristicas
    .groupby("characteristicGroup")
    .size()
    .sort_values(ascending=False)
)

display(grupos.head(15))

characteristicGroup
Organics, Other                  2269
Organics, Pesticide              2084
Population/Community             1999
Inorganics, Minor, Metals        1004
Organics, PCBs                    985
Information                       663
Organics, PFAS                    464
Physical                          460
Sediment                          408
Microbiological                   270
Radiochemical                     269
Inorganics, Major, Non-metals     256
Inorganics, Minor, Non-metals     184
Nutrient                          171
Inorganics, Major, Metals         152
dtype: int64

### 5.1. Resultado e análise do agrupamento

O agrupamento dos registros pela coluna `characteristicGroup` foi executado com sucesso.

A operação identificou a quantidade de registros em cada grupo e organizou os resultados em ordem decrescente.

#### Principais resultados

Os três grupos com maior quantidade de registros foram:

- Organics, Other: 2.269 registros.
- Organics, Pesticide: 2.084 registros.
- Population/Community: 1.999 registros.

#### Análise técnica

A operação `groupby()` combinada com `size()` permitiu realizar uma agregação dos registros sem necessidade de percorrer individualmente as linhas do DataFrame.

O resultado foi armazenado em uma Series Pandas e ordenado utilizando `sort_values()`.

É importante destacar que a contagem representa o número de registros por grupo, não a quantidade de características únicas.

#### Conclusão

A análise permitiu identificar a distribuição dos registros entre os grupos de características ambientais disponibilizados pela API USGS.

Essa exploração inicial fornece uma visão da composição do catálogo e prepara os dados para análises mais específicas sobre nomes de características, códigos de parâmetros e possíveis duplicidades.

## 6. Seleção e filtragem de características de interesse

### Objetivo

Selecionar os grupos de características ambientais considerados relevantes para a análise da qualidade da água e filtrar os registros correspondentes no DataFrame original.

### Justificativa

O catálogo da API USGS disponibiliza uma grande variedade de características ambientais, mas nem todas são relevantes para o escopo deste projeto.

Para direcionar a investigação, definimos uma lista de grupos relacionados à qualidade da água e utilizamos essa lista como critério de seleção.

Essa abordagem permite reduzir o conjunto de dados e concentrar as próximas análises nas características de maior interesse para o projeto.

### Grupos selecionados

- Physical
- Microbiological
- Nutrient
- Inorganics, Major, Non-metals
- Inorganics, Major, Metals
- Radiochemical

### Conceitos técnicos

**Lista Python (`list`):** estrutura utilizada para armazenar uma coleção ordenada de elementos.

**Filtro booleano:** operação que seleciona registros de acordo com condições que retornam `True` ou `False`.

**isin():** método do Pandas que verifica se os valores de uma coluna pertencem a uma lista ou coleção de referência.

**copy():** método que cria uma cópia independente dos dados selecionados, evitando alterações acidentais no DataFrame original.

**sort_values():** método utilizado para ordenar os registros por uma ou mais colunas.

### Explicação dos comandos

- `grupos_interesse = [...]`: define os grupos que serão considerados na análise.
- `df_caracteristicas["characteristicGroup"]`: acessa a coluna utilizada como critério de filtragem.
- `.isin(grupos_interesse)`: verifica quais registros pertencem aos grupos selecionados.
- `df_caracteristicas[condição]`: retorna somente os registros que atendem ao filtro.
- `.copy()`: cria uma cópia independente do subconjunto selecionado.
- `len(df_selecao)`: contabiliza os registros selecionados.
- `df_selecao[[...]]`: seleciona somente as colunas necessárias para visualização.
- `.sort_values(["characteristicGroup", "characteristicName"])`: ordena os registros pelo grupo e pelo nome da característica.
- `display()`: apresenta o resultado no notebook.

In [7]:
# Grupos de interesse para qualidade da água

grupos_interesse = [
    "Physical",
    "Microbiological",
    "Nutrient",
    "Inorganics, Major, Non-metals",
    "Inorganics, Major, Metals",
    "Radiochemical"
]

# Filtrar características
df_selecao = df_caracteristicas[
    df_caracteristicas["characteristicGroup"].isin(grupos_interesse)
].copy()

# Visualizar resultado
print("Total de características selecionadas:", len(df_selecao))

display(
    df_selecao[
        [
            "characteristicGroup",
            "parameterCode",
            "characteristicName"
        ]
    ].sort_values(
        ["characteristicGroup", "characteristicName"]
    )
)

Total de características selecionadas: 1578


,characteristicGroup,parameterCode,characteristicName
2607,"Inorganics, Major, Metals",00910,Calcium
2608,"Inorganics, Major, Metals",62456,Calcium
2609,"Inorganics, Major, Metals",30239,Calcium
2610,"Inorganics, Major, Metals",52713,Calcium
2611,"Inorganics, Major, Metals",01476,Calcium
...,...,...,...
11843,Radiochemical,82889,Xenon-136
11844,Radiochemical,85634,Xenon-136
11891,Radiochemical,29301,Zinc-65
11926,Radiochemical,27701,Zirconium-95


### 6.1. Resultado e análise da seleção

A filtragem dos registros foi executada com sucesso.

**Resultado:** 1.578 registros selecionados a partir dos 11.946 registros originalmente disponíveis.

#### Critérios utilizados

Foram selecionados seis grupos de características ambientais relacionados ao escopo da análise de qualidade da água.

A filtragem foi realizada utilizando o método `isin()`, que verifica a correspondência dos valores da coluna `characteristicGroup` com os elementos da lista `grupos_interesse`.

#### Análise técnica

O resultado foi armazenado em um novo DataFrame denominado `df_selecao`.

A utilização de `.copy()` cria uma cópia independente do subconjunto selecionado, permitindo realizar transformações posteriores sem modificar diretamente o DataFrame original.

Para facilitar a inspeção, foram apresentadas três colunas e os registros foram ordenados alfabeticamente pelo grupo e pelo nome da característica.

#### Conclusão

A seleção reduziu o conjunto de dados de 11.946 para 1.578 registros, concentrando a investigação em seis grupos considerados relevantes para o projeto.

O DataFrame filtrado será utilizado nas próximas etapas para investigar características específicas, códigos de parâmetros e possíveis duplicidades.

## 7. Pesquisa e identificação de parâmetros de qualidade da água

### Objetivo

Identificar registros relacionados a parâmetros de interesse para a análise da qualidade da água, utilizando palavras-chave e expressões regulares para pesquisar em diferentes colunas do DataFrame filtrado.

### Justificativa

Após selecionar os grupos de características ambientais relevantes, precisamos identificar quais registros correspondem aos parâmetros que pretendemos investigar.

Como os nomes e as descrições das características podem apresentar variações textuais, utilizaremos expressões regulares para ampliar a capacidade de pesquisa.

A pesquisa será realizada nas colunas `characteristicName` e `characteristicNameUserSupplied`, considerando tanto o nome da característica quanto sua descrição detalhada.

### Termos pesquisados

- pH
- dissolved oxygen
- temperature
- conductivity
- turbidity
- nitrate
- phosphorus
- e. coli

### Conceitos técnicos

**Expressões regulares (RegEx):** padrões utilizados para localizar, identificar e manipular sequências de caracteres em textos.

**re.escape():** função do módulo `re` que transforma caracteres especiais em caracteres literais, evitando que sejam interpretados como operadores de expressão regular.

**re.IGNORECASE / case=False:** permite realizar pesquisas sem diferenciar letras maiúsculas e minúsculas.

**str.contains():** método do Pandas que verifica se determinado padrão textual está presente em cada elemento de uma coluna.

**Máscara booleana:** estrutura de valores `True` e `False` utilizada para selecionar registros.

**Operador OR (`|`):** combina condições, permitindo selecionar registros que atendam a pelo menos uma delas.

**nunique():** método que contabiliza a quantidade de valores distintos de uma coluna, desconsiderando valores ausentes por padrão.

### Explicação dos comandos

- `termos = [...]`: define as palavras-chave utilizadas na pesquisa.
- `padroes = [...]`: inicializa a lista de padrões RegEx.
- `(?<![A-Za-z])pH(?![A-Za-z])`: procura pH sem que existam letras imediatamente antes ou depois.
- `termos[1:]`: seleciona os elementos da lista a partir do segundo, excluindo pH.
- `re.escape(termo)`: protege caracteres especiais presentes nos termos pesquisados.
- `padroes.extend()`: acrescenta novos padrões à lista existente.
- `"|".join(padroes)`: combina os padrões utilizando o operador OR das expressões regulares.
- `.str.contains()`: pesquisa o padrão nas colunas selecionadas.
- `case=False`: ignora diferenças entre letras maiúsculas e minúsculas.
- `na=False`: considera valores ausentes como não correspondentes à pesquisa.
- `df_selecao.loc[mascara]`: seleciona os registros que atendem aos critérios.
- `.copy()`: cria uma cópia independente dos registros selecionados.
- `.nunique()`: identifica a quantidade de códigos distintos.

### Resultado esperado

Um novo DataFrame contendo os registros relacionados aos termos pesquisados, acompanhado da quantidade total de registros e da quantidade de códigos de parâmetros distintos.

In [8]:
# Pesquisa de parâmetros de qualidade da água

termos = [
    "pH",
    "dissolved oxygen",
    "temperature",
    "conductivity",
    "turbidity",
    "nitrate",
    "phosphorus",
    "e. coli"
]

# Construção dos padrões
padroes = [
    r"(?<![A-Za-z])pH(?![A-Za-z])"
]

padroes.extend(
    rf"(?<!\w){re.escape(termo)}(?!\w)"
    for termo in termos[1:]
)

padrao = "|".join(padroes)

# Pesquisa nas colunas
mascara = (
    df_selecao["characteristicName"].str.contains(
        padrao, case=False, na=False
    )
    |
    df_selecao["characteristicNameUserSupplied"].str.contains(
        padrao, case=False, na=False
    )
)

# Criação do DataFrame filtrado
df_parametros = df_selecao.loc[mascara].copy()

print("Total de registros:", len(df_parametros))
print("Códigos distintos:", df_parametros["parameterCode"].nunique())

display(df_parametros.head(30))

Total de registros: 240
Códigos distintos: 221


,characteristicGroup,parameterCode,characteristicName,characteristicNameUserSupplied
1097,"Inorganics, Major, Non-metals",00410,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1098,"Inorganics, Major, Non-metals",95410,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1099,"Inorganics, Major, Non-metals",00417,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1100,"Inorganics, Major, Non-metals",90410,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1105,"Inorganics, Major, Non-metals",00411,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1106,"Inorganics, Major, Non-metals",00415,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1217,"Inorganics, Major, Non-metals",00418,Alkalinity,"Alkalinity as calcium carbonate, water, filter..."
1218,"Inorganics, Major, Non-metals",39036,Alkalinity,"Alkalinity as calcium carbonate, water, filter..."
1219,"Inorganics, Major, Non-metals",00421,Alkalinity,"Alkalinity as calcium carbonate, water, filter..."
1220,"Inorganics, Major, Non-metals",29801,Alkalinity,"Alkalinity as calcium carbonate, water, filter..."


### 7.1. Resultado e análise da pesquisa de parâmetros

A pesquisa textual foi executada com sucesso nas colunas `characteristicName` e `characteristicNameUserSupplied`.

**Resultados obtidos:**

- Registros encontrados: 240.
- Códigos distintos: 221.
- Termos pesquisados: 8.

#### Análise técnica

A utilização de expressões regulares permitiu combinar diferentes critérios de pesquisa em um único padrão.

A máscara booleana foi construída considerando correspondências em duas colunas, utilizando o operador OR.

Os registros encontrados foram armazenados no DataFrame `df_parametros`, preservando a estrutura original dos dados.

#### Observação

A diferença entre registros encontrados e códigos distintos indica a necessidade de investigar possíveis repetições de códigos e valores ausentes.

A quantidade de códigos distintos não deve ser interpretada como quantidade definitiva de parâmetros de qualidade da água.

#### Conclusão

A pesquisa identificou 240 registros relacionados aos termos definidos, contendo 221 códigos distintos.

O resultado fornece uma primeira seleção de características para investigação mais detalhada, permitindo avançar para a análise de correspondências, duplicidades e consistência dos códigos de parâmetros.

## 8. Diagnóstico individual das correspondências

### Objetivo

Investigar individualmente a quantidade de registros encontrados para cada termo utilizado na pesquisa de parâmetros de qualidade da água.

### Justificativa

Na etapa anterior, os oito termos foram combinados em uma única expressão regular, resultando em 240 registros selecionados.

Entretanto, essa abordagem não permite identificar diretamente quantas correspondências foram encontradas para cada termo.

Para solucionar essa limitação, será executado um diagnóstico individual, percorrendo a lista de termos e aplicando cada padrão de pesquisa separadamente.

### Estratégia

Para cada termo:

1. Verificar se o termo corresponde a pH.
2. Construir o padrão de expressão regular apropriado.
3. Pesquisar nas colunas `characteristicName` e `characteristicNameUserSupplied`.
4. Combinar as condições utilizando o operador OR.
5. Contabilizar as correspondências encontradas.
6. Exibir o resultado individualmente.

### Conceitos técnicos

**Laço `for`:** estrutura de repetição que percorre os elementos de uma coleção.

**Condicional `if/else`:** estrutura utilizada para executar diferentes instruções conforme uma condição.

**Máscara booleana:** conjunto de valores `True` e `False` que representa quais registros atendem aos critérios de pesquisa.

**Método `sum()`:** soma os valores da máscara booleana. Como `True` equivale a 1 e `False` equivale a 0, o resultado representa a quantidade de correspondências.

**Interpolação de strings (`f-string`):** permite inserir variáveis diretamente em textos utilizando o prefixo `f`.

### Resultado esperado

Uma listagem contendo a quantidade de correspondências encontradas individualmente para cada um dos oito termos pesquisados.

Essa análise também permite identificar possíveis sobreposições entre os critérios de pesquisa.

In [9]:
# Diagnóstico das correspondências

for termo in termos:

    if termo == "pH":
        padrao_teste = r"(?<![A-Za-z])pH(?![A-Za-z])"
    else:
        padrao_teste = rf"(?<!\w){re.escape(termo)}(?!\w)"

    resultado = (
        df_selecao["characteristicName"].str.contains(
            padrao_teste, case=False, na=False
        )
        |
        df_selecao["characteristicNameUserSupplied"].str.contains(
            padrao_teste, case=False, na=False
        )
    )

    print(f"{termo}: {resultado.sum()} correspondências")

pH: 46 correspondências
dissolved oxygen: 11 correspondências
temperature: 39 correspondências
conductivity: 0 correspondências
turbidity: 26 correspondências
nitrate: 54 correspondências
phosphorus: 61 correspondências
e. coli: 4 correspondências


### 8.1. Resultado e análise do diagnóstico

O diagnóstico individual dos oito termos foi executado com sucesso.

#### Resultados obtidos

| Termo | Correspondências |
|---|---:|
| pH | 46 |
| dissolved oxygen | 11 |
| temperature | 39 |
| conductivity | 0 |
| turbidity | 26 |
| nitrate | 54 |
| phosphorus | 61 |
| e. coli | 4 |

A soma das correspondências individuais foi de 241, enquanto a pesquisa combinada da etapa anterior identificou 240 registros.

#### Análise técnica

A diferença entre os resultados indica uma possível sobreposição entre os critérios de pesquisa.

Isso ocorre porque um mesmo registro pode conter mais de um termo pesquisado. Na contagem individual, ele participa da contabilização de cada termo correspondente. Na pesquisa combinada, entretanto, o registro é selecionado apenas uma vez.

O diagnóstico também identificou ausência de correspondências para o termo `conductivity`, indicando a necessidade de investigar possíveis variações de nomenclatura.

#### Conclusão

A análise individual permitiu identificar a distribuição das correspondências entre os termos pesquisados e evidenciou a necessidade de investigar sobreposições e possíveis diferenças de nomenclatura.

Esses resultados serão utilizados para orientar as próximas etapas de validação e preparação dos dados.

## 9. Inspeção dos registros de Alcalinidade (Alkalinity)

### Objetivo

Investigar os registros identificados com o nome de característica `Alkalinity`, verificando os códigos de parâmetros associados e suas respectivas descrições.

### Justificativa

Durante a pesquisa de parâmetros da etapa anterior, foram encontrados registros que compartilham o mesmo nome de característica, mas apresentam códigos de parâmetros diferentes.

Para compreender essa ocorrência, será realizada uma inspeção específica dos registros de alcalinidade.

A análise permitirá verificar se os diferentes códigos representam variações de descrição, condições de medição ou outras classificações disponibilizadas pela API USGS.

### Estratégia

1. Filtrar o DataFrame `df_parametros` utilizando a coluna `characteristicName`.
2. Selecionar os registros cujo nome seja exatamente `Alkalinity`.
3. Criar um novo DataFrame independente.
4. Exibir os códigos e as descrições associadas à característica.

### Conceitos técnicos

**Comparação de igualdade (`==`):** verifica se o valor de uma coluna corresponde exatamente ao valor especificado.

**Filtro booleano:** seleciona os registros que atendem à condição estabelecida.

**`.loc[]`:** permite acessar e selecionar linhas e colunas de um DataFrame utilizando rótulos e condições.

**`.copy()`:** cria uma cópia independente dos registros selecionados.

**Seleção de colunas:** permite restringir a visualização aos campos relevantes para a investigação.

### Resultado esperado

Uma relação dos registros de alcalinidade contendo seus códigos de parâmetros e descrições detalhadas, permitindo investigar a existência de diferentes representações da mesma característica.

In [10]:
# Inspecionar registros de Alkalinity

df_diagnostico = df_parametros[
    df_parametros["characteristicName"] == "Alkalinity"
].copy()

display(
    df_diagnostico[
        [
            "parameterCode",
            "characteristicName",
            "characteristicNameUserSupplied"
        ]
    ]
)

,parameterCode,characteristicName,characteristicNameUserSupplied
1097,00410,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1098,95410,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1099,00417,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1100,90410,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1105,00411,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1106,00415,Alkalinity,Acid neutralizing capacity (ANC) as calcium ca...
1217,00418,Alkalinity,"Alkalinity as calcium carbonate, water, filter..."
1218,39036,Alkalinity,"Alkalinity as calcium carbonate, water, filter..."
1219,00421,Alkalinity,"Alkalinity as calcium carbonate, water, filter..."
1220,29801,Alkalinity,"Alkalinity as calcium carbonate, water, filter..."


### 9.1. Resultado e análise dos registros de Alkalinity

A inspeção dos registros de alcalinidade foi executada com sucesso.

**Resultado:** foram identificados 10 registros com o nome de característica `Alkalinity`.

#### Principais observações

Todos os registros apresentam o mesmo nome padronizado, mas estão associados a diferentes códigos de parâmetros e descrições detalhadas.

As descrições apresentam referências a diferentes formas de representação da alcalinidade, incluindo capacidade de neutralização ácida (ANC), carbonato de cálcio e condições de filtragem da água.

#### Análise técnica

A investigação demonstra que a coluna `characteristicName`, isoladamente, não é suficiente para determinar a equivalência entre parâmetros.

É necessário considerar também o código do parâmetro e sua descrição detalhada.

Outro cuidado importante é preservar os códigos como texto, mantendo zeros à esquerda e a representação original dos identificadores.

#### Conclusão

A inspeção permitiu identificar diferentes códigos associados à característica `Alkalinity`.

Os resultados demonstram a necessidade de investigar a semântica dos códigos e suas descrições antes de estabelecer regras de padronização, agrupamento ou eliminação de registros.

Essa análise contribui para a definição de critérios de qualidade e consistência dos dados no pipeline ETL.

## 10. Investigação das correspondências de Alkalinity

### Objetivo

Investigar quais termos de pesquisa identificaram os registros de alcalinidade selecionados anteriormente, verificando em qual coluna textual ocorreram as correspondências.

### Justificativa

Na Célula 09, foram identificados dez registros com `characteristicName = Alkalinity`.

Entretanto, esses registros foram selecionados durante a pesquisa de parâmetros da Célula 07, que utiliza diferentes termos relacionados à qualidade da água.

Precisamos identificar por que registros de alcalinidade foram incluídos na seleção e verificar se a correspondência ocorreu no nome padronizado ou na descrição detalhada.

Essa investigação ajuda a identificar falsos positivos produzidos por pesquisas textuais e a avaliar a precisão dos critérios de seleção.

### Estratégia

1. Percorrer simultaneamente a lista de termos e seus respectivos padrões RegEx.
2. Verificar cada termo nas colunas `characteristicName` e `characteristicNameUserSupplied`.
3. Construir uma máscara booleana para cada pesquisa.
4. Verificar se existem correspondências utilizando `any()`.
5. Contabilizar as ocorrências utilizando `sum()`.
6. Exibir os registros encontrados e seus respectivos códigos.

### Conceitos técnicos

**zip():** função Python que combina elementos de duas ou mais coleções, permitindo percorrê-los simultaneamente.

**for aninhado:** estrutura de repetição dentro de outra estrutura de repetição.

**any():** verifica se pelo menos um elemento de uma coleção é verdadeiro.

**sum():** contabiliza os valores verdadeiros de uma máscara booleana.

**to_string():** converte um DataFrame em representação textual, permitindo controlar a apresentação dos índices.

**Regex:** mecanismo utilizado para identificar padrões de texto.

### Resultado esperado

Identificar quais termos produziram correspondências nos registros de alcalinidade e em quais colunas essas correspondências ocorreram.

In [11]:
# Investigar correspondências de Alkalinity

for termo, padrao_teste in zip(termos, padroes):

    for coluna in [
        "characteristicName",
        "characteristicNameUserSupplied"
    ]:

        resultado = df_diagnostico[coluna].str.contains(
            padrao_teste,
            case=False,
            na=False,
            regex=True
        )

        if resultado.any():

            print(f"\nTermo: {termo}")
            print(f"Coluna: {coluna}")
            print("Quantidade:", resultado.sum())

            display(
                df_diagnostico.loc[
                    resultado,
                    [coluna, "parameterCode"]
                ].to_string(index=False)
            )


Termo: pH
Coluna: characteristicNameUserSupplied
Quantidade: 10


'                                                                                                characteristicNameUserSupplied parameterCode\n       Acid neutralizing capacity (ANC) as calcium carbonate, water, unfiltered, fixed endpoint (EP) (pH 4.5) titration, field         00410\n  Acid neutralizing capacity (ANC) as calcium carbonate, water, unfiltered, fixed endpoint (EP) (pH 4.5) titration, laboratory         95410\n  Acid neutralizing capacity (ANC) as calcium carbonate, water, unfiltered, fixed endpoint (EP) (pH 4.5) titration, laboratory         00417\n  Acid neutralizing capacity (ANC) as calcium carbonate, water, unfiltered, fixed endpoint (EP) (pH 4.5) titration, laboratory         90410\n  Acid neutralizing capacity (ANC) as calcium carbonate, water, unfiltered, methyl orange endpoint (EP) (pH 3.1-4.4) titration         00411\nAcid neutralizing capacity (ANC) as calcium carbonate, water, unfiltered, phenolphthalein endpoint (EP) (pH 8.5-9.0) titration         00415\n     

### 10.1. Resultado e análise das correspondências de Alkalinity

A investigação das correspondências foi executada com sucesso.

#### Resultado identificado

Foram encontradas dez correspondências para o termo `pH`, todas na coluna `characteristicNameUserSupplied`.

Não foram identificadas correspondências para os demais termos pesquisados nos registros de alcalinidade.

#### Análise técnica

A investigação demonstrou que os registros de `Alkalinity` foram incluídos na seleção anterior porque suas descrições detalhadas contêm referências a procedimentos de titulação que utilizam valores de pH como pontos finais.

Isso caracteriza um possível falso positivo na pesquisa textual.

A característica principal continua sendo alcalinidade, embora sua descrição contenha referências a pH.

#### Conclusão

A análise evidenciou uma limitação da identificação de parâmetros baseada exclusivamente em correspondências textuais.

Para aumentar a precisão da seleção, será necessário considerar o nome padronizado da característica, os códigos de parâmetros e as descrições detalhadas em conjunto.

Essa investigação reforça a importância da validação semântica dos dados antes de sua utilização em transformações e análises posteriores.

## 11. Pesquisa refinada de parâmetros

### Objetivo

Refinar a identificação dos parâmetros relacionados à qualidade da água, utilizando exclusivamente a coluna `characteristicName` como critério de pesquisa.

### Justificativa

Na investigação anterior, identificamos que a pesquisa realizada também na coluna `characteristicNameUserSupplied` produzia correspondências que não representavam necessariamente a característica principal do registro.

Um exemplo foi a identificação de registros de `Alkalinity` por conterem referências a pH nas descrições de procedimentos de titulação.

Para reduzir esse tipo de ocorrência, a estratégia será modificada.

A pesquisa passará a considerar somente os nomes padronizados das características, evitando que referências secundárias presentes nas descrições detalhadas determinem a seleção.

### Estratégia

1. Definir expressões regulares para os oito parâmetros de interesse.
2. Considerar variações de nomenclatura para condutividade.
3. Construir uma expressão regular combinada utilizando o operador OR.
4. Pesquisar exclusivamente na coluna `characteristicName`.
5. Criar uma cópia independente dos registros selecionados.
6. Contabilizar os registros encontrados e os códigos distintos.
7. Exibir uma amostra de até 50 registros.

### Conceitos técnicos

**Expressões regulares (RegEx):** permitem identificar padrões específicos em textos.

**Alternância (`|`):** permite pesquisar diferentes padrões utilizando uma única expressão regular.

**str.contains():** identifica valores que contêm determinado padrão textual.

**case=False:** torna a pesquisa independente de letras maiúsculas ou minúsculas.

**na=False:** trata valores ausentes como não correspondentes.

**nunique():** contabiliza a quantidade de valores distintos de uma coluna, desconsiderando valores ausentes por padrão.

**head(50):** retorna os primeiros 50 registros do DataFrame.

### Resultado esperado

Uma seleção refinada dos parâmetros de interesse, com menor interferência de palavras encontradas apenas nas descrições complementares.

In [12]:
# Pesquisa refinada de parâmetros

padroes_nome = [
    r"(?<![A-Za-z])pH(?![A-Za-z])",
    r"dissolved oxygen",
    r"temperature",
    r"conductivity|specific conductance",
    r"turbidity",
    r"nitrate",
    r"phosphorus",
    r"e\. coli"
]

padrao_nome = "|".join(padroes_nome)

# Pesquisar somente no nome da característica
mascara = df_selecao["characteristicName"].str.contains(
    padrao_nome,
    case=False,
    na=False,
    regex=True
)

# Criar DataFrame independente
df_parametros = df_selecao.loc[mascara].copy()

# Exibir resultados
print("Total de registros:", len(df_parametros))
print(
    "Códigos distintos:",
    df_parametros["parameterCode"].nunique()
)

display(
    df_parametros[
        [
            "characteristicGroup",
            "parameterCode",
            "characteristicName",
            "characteristicNameUserSupplied"
        ]
    ].head(50)
)

Total de registros: 176
Códigos distintos: 158


,characteristicGroup,parameterCode,characteristicName,characteristicNameUserSupplied
4211,"Inorganics, Major, Non-metals",90300,Dissolved oxygen (DO),"Dissolved oxygen (DO), area-weighted mean, wat..."
4212,"Inorganics, Major, Non-metals",00301,Dissolved oxygen saturation,"Dissolved oxygen (DO) saturation, water, unfil..."
4213,"Inorganics, Major, Non-metals",NaN,Dissolved oxygen saturation,"Dissolved oxygen (DO) saturation, water, unfil..."
4214,"Inorganics, Major, Non-metals",72210,Dissolved oxygen saturation,"Dissolved oxygen (DO) saturation, water, unfil..."
4215,"Inorganics, Major, Non-metals",85550,Dissolved oxygen (DO),"Dissolved oxygen (DO) (volume basis), dissolve..."
4216,"Inorganics, Major, Non-metals",NaN,Dissolved oxygen (DO),"Dissolved oxygen (DO), water, unfiltered"
4217,"Inorganics, Major, Non-metals",00300,Dissolved oxygen (DO),"Dissolved oxygen (DO), water, unfiltered"
4218,"Inorganics, Major, Non-metals",NaN,Dissolved oxygen (DO),"Dissolved oxygen (DO), water, unfiltered, labo..."
4219,"Inorganics, Major, Non-metals",62971,Dissolved oxygen (DO),"Dissolved oxygen (DO), water, unfiltered, labo..."
4220,"Inorganics, Major, Non-metals",99985,Dissolved oxygen (DO),"Dissolved oxygen (DO), water, unfiltered, mid-..."


### 11.1. Resultado e análise da pesquisa refinada

A pesquisa refinada dos parâmetros foi executada com sucesso.

#### Resultados obtidos

- Total de registros selecionados: 176.
- Códigos distintos: 158.

#### Alteração da estratégia

A principal alteração foi restringir a pesquisa à coluna `characteristicName`, deixando de utilizar a descrição detalhada como critério de seleção.

Essa mudança foi motivada pela identificação de possíveis falsos positivos na investigação anterior, especialmente registros de alcalinidade identificados pela presença de referências a pH em suas descrições.

Também foram incorporadas variações de nomenclatura para condutividade, contemplando `conductivity` e `specific conductance`.

#### Análise técnica

A pesquisa refinada reduziu o resultado de 240 para 176 registros.

A amostra apresenta diferentes características relacionadas aos grupos físicos, microbiológicos, nutricionais e inorgânicos.

Também foram identificados registros sem código de parâmetro (`NaN`), indicando a necessidade de uma investigação adicional sobre a completude dos identificadores.

#### Conclusão

A nova estratégia tornou o critério de seleção mais restritivo e reduziu a influência de descrições complementares sobre a identificação dos parâmetros.

Entretanto, os resultados ainda precisam passar por validações de integridade, unicidade e correspondência semântica antes de serem considerados um catálogo definitivo de parâmetros de qualidade da água.

## 12. Análise de cardinalidade dos parâmetros

### Objetivo

Avaliar a cardinalidade, a completude e a distribuição dos parâmetros identificados na pesquisa refinada, verificando a quantidade de registros, nomes distintos, códigos distintos e identificadores ausentes.

### Justificativa

Após a seleção refinada dos 176 registros, é necessário compreender melhor a estrutura e a qualidade do conjunto de dados.

A existência de diferentes registros associados à mesma característica pode indicar variações de parâmetros, condições de medição ou representações distintas.

Além disso, foram identificados códigos ausentes, tornando necessária uma análise da completude dos identificadores.

### Estratégia

1. Contabilizar o total de registros.
2. Identificar a quantidade de nomes distintos em `characteristicName`.
3. Identificar a quantidade de códigos distintos em `parameterCode`.
4. Contabilizar os códigos ausentes.
5. Agrupar os registros por grupo e nome da característica.
6. Calcular métricas de quantidade de registros, códigos distintos e códigos ausentes.
7. Ordenar o resultado pela quantidade de registros, em ordem decrescente.

### Conceitos técnicos

**Cardinalidade:** representa a quantidade de valores distintos existentes em uma variável ou coluna.

**nunique():** contabiliza valores distintos, desconsiderando valores ausentes por padrão.

**isna():** identifica valores ausentes, retornando uma máscara booleana.

**groupby():** agrupa registros de acordo com uma ou mais colunas.

**dropna=False:** permite manter grupos cujas chaves possuem valores ausentes.

**agg():** executa múltiplas operações de agregação sobre os grupos.

**size:** contabiliza a quantidade de linhas de cada grupo.

**lambda:** permite definir uma função anônima para executar uma operação específica.

**reset_index():** transforma os índices dos grupos em colunas do DataFrame.

**sort_values():** ordena os registros de acordo com uma ou mais colunas.

### Resultado esperado

Um diagnóstico quantitativo dos parâmetros selecionados, permitindo identificar características com maior quantidade de registros, variações de códigos e problemas de completude dos identificadores.

In [13]:
# Análise de cardinalidade dos parâmetros

print("Registros:", len(df_parametros))

print(
    "Nomes distintos:",
    df_parametros["characteristicName"].nunique()
)

print(
    "Códigos distintos:",
    df_parametros["parameterCode"].nunique()
)

print(
    "Códigos ausentes:",
    df_parametros["parameterCode"].isna().sum()
)

# Distribuição por nome da característica
resumo_parametros = (
    df_parametros
    .groupby(
        ["characteristicGroup", "characteristicName"],
        dropna=False
    )
    .agg(
        registros=("parameterCode", "size"),
        codigos_distintos=("parameterCode", "nunique"),
        codigos_ausentes=("parameterCode", lambda x: x.isna().sum())
    )
    .reset_index()
    .sort_values("registros", ascending=False)
)

display(resumo_parametros)

Registros: 176
Nomes distintos: 24
Códigos distintos: 158
Códigos ausentes: 14


,characteristicGroup,characteristicName,registros,codigos_distintos,codigos_ausentes
12,Nutrient,Phosphorus,40,39,1
21,Physical,Turbidity,25,21,2
9,Nutrient,Nitrate,21,21,0
23,Physical,pH,18,15,3
13,Physical,Specific conductance,15,13,2
0,"Inorganics, Major, Non-metals",Dissolved oxygen (DO),8,6,2
7,Nutrient,Inorganic nitrogen (nitrate and nitrite),8,7,1
19,Physical,"Temperature, water",6,5,1
14,Physical,Temperature,5,5,0
15,Physical,"Temperature, air",4,3,1


### 12.1. Resultado e análise da cardinalidade dos parâmetros

A análise de cardinalidade foi executada com sucesso.

#### Indicadores gerais

| Indicador | Resultado |
|---|---:|
| Registros analisados | 176 |
| Nomes distintos | 24 |
| Códigos distintos | 158 |
| Códigos ausentes | 14 |

A completude dos códigos de parâmetros foi de aproximadamente 92,05%.

#### Principais características identificadas

A característica `Phosphorus` apresentou a maior quantidade de registros, totalizando 40 ocorrências.

Em seguida, destacaram-se `Turbidity`, com 25 registros, e `Nitrate`, com 21 registros.

A análise também identificou diferenças entre a quantidade de registros e a quantidade de códigos distintos, evidenciando a existência de identificadores repetidos em determinadas características.

#### Análise de qualidade

Foram identificados 14 registros sem código de parâmetro.

A característica `Nitrate` apresentou correspondência entre quantidade de registros e códigos distintos, sem identificadores ausentes.

Já `Turbidity`, `pH` e `Dissolved oxygen (DO)` apresentaram situações que merecem investigação adicional.

#### Conclusão

O perfilamento permitiu avaliar a cardinalidade, a distribuição e a completude dos parâmetros selecionados.

Os resultados demonstram que a quantidade de registros não representa necessariamente a quantidade de parâmetros únicos.

A identificação de códigos ausentes e repetidos fornece subsídios para as próximas etapas de validação e preparação dos dados.

Os identificadores ausentes deverão ser investigados antes de qualquer decisão de exclusão ou substituição.

## 13. Exibição completa das características encontradas

### Objetivo

Apresentar integralmente o resumo estatístico das características identificadas na pesquisa refinada de parâmetros da API USGS.

### Justificativa

Na etapa anterior, foi construído o DataFrame `resumo_parametros`, contendo a distribuição dos registros por grupo e nome da característica.

A visualização convencional de DataFrames no Jupyter Notebook pode limitar a quantidade de linhas apresentadas, dependendo das configurações de exibição.

Para garantir que todas as características identificadas sejam apresentadas, será utilizada a conversão do DataFrame para uma representação textual.

### Estratégia

1. Recuperar o DataFrame `resumo_parametros`, criado na Célula 12.
2. Converter seu conteúdo para uma representação textual utilizando `to_string()`.
3. Desabilitar a exibição dos índices com `index=False`.
4. Apresentar integralmente os resultados no notebook.

### Conceitos técnicos

**to_string():** converte um DataFrame em uma representação textual, permitindo controlar a apresentação dos dados.

**index=False:** impede que os índices do DataFrame sejam apresentados na saída.

**print():** exibe o conteúdo textual no notebook.

### Resultado esperado

Exibição completa das 24 características identificadas, incluindo grupo, quantidade de registros, códigos distintos e códigos ausentes.

In [14]:
# Exibir todas as características encontradas

print(
    resumo_parametros.to_string(index=False)
)

          characteristicGroup                           characteristicName  registros  codigos_distintos  codigos_ausentes
                     Nutrient                                   Phosphorus         40                 39                 1
                     Physical                                    Turbidity         25                 21                 2
                     Nutrient                                      Nitrate         21                 21                 0
                     Physical                                           pH         18                 15                 3
                     Physical                         Specific conductance         15                 13                 2
Inorganics, Major, Non-metals                        Dissolved oxygen (DO)          8                  6                 2
                     Nutrient     Inorganic nitrogen (nitrate and nitrite)          8                  7                 1
                

### 13.1. Resultado e conclusão da exploração

A apresentação completa do resumo estatístico foi executada com sucesso.

#### Resultado final

Foram apresentadas integralmente as 24 características identificadas durante a pesquisa refinada, totalizando 176 registros e 158 códigos distintos.

A tabela também apresenta a quantidade de códigos ausentes e a distribuição dos registros por grupo e característica.

#### Análise técnica

A visualização completa permite identificar variações de nomenclatura dentro de um mesmo conjunto de parâmetros.

Um exemplo é a característica `Temperature`, que apresenta diferentes classificações relacionadas ao meio ou às condições de medição.

Essa informação será importante para futuras decisões de padronização e modelagem dos dados.

#### Conclusão

A etapa de exploração da API USGS foi concluída, permitindo identificar o endpoint de características, inspecionar a estrutura JSON, transformar os registros em DataFrame e investigar os parâmetros relacionados à qualidade da água.

O resultado constitui uma base inicial para as próximas etapas do projeto Open Water Analytics, especialmente a definição dos critérios de seleção, validação e transformação dos dados.

A exploração não representa ainda a validação definitiva dos parâmetros ambientais, sendo necessária uma investigação complementar sobre os códigos e suas respectivas definições.

## 14. Descoberta das estações de monitoramento USGS

### Objetivo

Identificar estações de monitoramento disponibilizadas pela API USGS, investigando seus identificadores, informações geográficas e coordenadas.

### Justificativa

Após explorar o catálogo de características da USGS, precisamos identificar as estações responsáveis pelo monitoramento ambiental.

As coordenadas geográficas serão importantes para estabelecer uma relação espacial entre as estações de monitoramento e os dados meteorológicos disponibilizados pela Open-Meteo.

### Estratégia

1. Consultar o endpoint de estações de monitoramento da USGS.
2. Configurar a quantidade de registros retornados.
3. Executar a requisição HTTP utilizando a biblioteca Requests.
4. Validar a resposta recebida.
5. Converter o JSON em um objeto Python.
6. Transformar os registros GeoJSON em um DataFrame utilizando Pandas.
7. Inspecionar os campos e as informações retornadas.

### Conceitos técnicos

**GeoJSON:** formato utilizado para representar informações geográficas estruturadas em JSON.

**requests.get():** realiza uma requisição HTTP GET.

**raise_for_status():** verifica se a resposta HTTP representa uma condição de erro.

**json():** converte a resposta JSON em estruturas Python.

**pd.json_normalize():** transforma estruturas JSON em formato tabular, facilitando a análise.

### Resultado esperado

Identificar as estações disponíveis e inspecionar suas informações geográficas, especialmente latitude e longitude, para utilização nas próximas etapas do projeto.

In [15]:
# Endpoint de estações de monitoramento USGS
BASE_URL = "https://api.waterdata.usgs.gov/ogcapi/v1/collections/monitoring-locations/items"

# Parâmetros da consulta
PARAMS = {
    "limit": 5
}

# Requisição HTTP
response_locais = requests.get(
    BASE_URL,
    params=PARAMS,
    timeout=30
)

# Validação da resposta
response_locais.raise_for_status()

# Conversão para objeto Python
dados_locais = response_locais.json()

# Inspeção da resposta
print("Status HTTP:", response_locais.status_code)
print("Tipo de retorno:", dados_locais.get("type"))
print("Registros retornados:", dados_locais.get("numberReturned"))
print("Total de registros encontrados:", dados_locais.get("numberMatched"))

# Conversão para DataFrame
df_locais = pd.json_normalize(dados_locais["features"])

# Inspeção dos campos disponíveis
print("\nColunas disponíveis:")
print(df_locais.columns.tolist())

# Visualização dos registros
display(df_locais.head())

Status HTTP: 200
Tipo de retorno: FeatureCollection
Registros retornados: 5
Total de registros encontrados: None

Colunas disponíveis:
['type', 'id', 'geometry', 'properties.id', 'properties.agency_code', 'properties.agency_name', 'properties.monitoring_location_number', 'properties.monitoring_location_name', 'properties.district_code', 'properties.country_code', 'properties.country_name', 'properties.state_code', 'properties.state_name', 'properties.county_code', 'properties.county_name', 'properties.minor_civil_division_code', 'properties.site_type_code', 'properties.site_type', 'properties.hydrologic_unit_code', 'properties.basin_code', 'properties.altitude', 'properties.altitude_accuracy', 'properties.altitude_method_code', 'properties.altitude_method_name', 'properties.vertical_datum', 'properties.vertical_datum_name', 'properties.horizontal_positional_accuracy_code', 'properties.horizontal_positional_accuracy', 'properties.horizontal_position_method_code', 'properties.horizontal_

,type,id,geometry,properties.id,properties.agency_code,properties.agency_name,properties.monitoring_location_number,properties.monitoring_location_name,properties.district_code,properties.country_code,...,properties.construction_date,properties.aquifer_code,properties.national_aquifer_code,properties.aquifer_type_code,properties.well_constructed_depth,properties.hole_constructed_depth,properties.depth_source_code,properties.revision_note,properties.revision_created,properties.revision_modified
0,Feature,AL012-90100100001,None,AL012-90100100001,AL012,Alabama Office of Water Resources,90100100001,Autauga County Water Authority,01,US,...,None,None,None,None,None,None,None,None,None,None
1,Feature,AL012-90100100002,None,AL012-90100100002,AL012,Alabama Office of Water Resources,90100100002,Autaugaville Water System,01,US,...,None,None,None,None,None,None,None,None,None,None
2,Feature,AL012-90100100003,None,AL012-90100100003,AL012,Alabama Office of Water Resources,90100100003,Billingsley Water System,01,US,...,None,None,None,None,None,None,None,None,None,None
3,Feature,AL012-90100100005,None,AL012-90100100005,AL012,Alabama Office of Water Resources,90100100005,Water Works Board Of Prattville,01,US,...,None,None,None,None,None,None,None,None,None,None
4,Feature,AL012-90100100006,None,AL012-90100100006,AL012,Alabama Office of Water Resources,90100100006,West Autauga Water Authority,01,US,...,None,None,None,None,None,None,None,None,None,None



### Conclusão — Descoberta das estações de monitoramento USGS

A consulta ao endpoint de estações de monitoramento da API USGS foi executada com sucesso, retornando cinco registros no formato GeoJSON (`FeatureCollection`).

A conversão dos registros para um DataFrame utilizando `pd.json_normalize()` permitiu identificar 46 colunas, incluindo identificadores, informações das agências responsáveis e nomes das localizações monitoradas.

Durante a análise inicial, foram observados os seguintes pontos:

- Os cinco registros retornados pertencem a sistemas de abastecimento de água do estado do Alabama.
- A agência identificada foi a Alabama Office of Water Resources.
- O campo `geometry` apresentou valores ausentes nos registros analisados.
- O atributo `numberMatched` não foi disponibilizado na resposta recebida.

**Considerações técnicas**

A consulta demonstrou como consumir uma API com retorno GeoJSON e normalizar estruturas JSON em tabelas utilizando Pandas.

Entretanto, os primeiros registros retornados não representam necessariamente as estações adequadas ao objetivo da análise. Será necessário refinar os critérios de consulta para identificar localizações de monitoramento relevantes e verificar a disponibilidade de suas coordenadas geográficas.

**Resultado da etapa**

Foi realizada a primeira exploração do endpoint de localizações de monitoramento da USGS, identificando a estrutura dos dados e a necessidade de aplicar filtros específicos nas próximas consultas.



## 15. Filtragem das estações vinculadas à agência USGS

### Objetivo
Refinar a consulta de localizações de monitoramento, retornando registros associados à agência USGS.

### Justificativa
A consulta anterior retornou registros de diferentes agências, incluindo a Alabama Office of Water Resources. Para avançar na exploração, precisamos restringir os resultados à agência USGS.

### Estratégia
1. Reutilizar o endpoint de localizações de monitoramento.
2. Aplicar o filtro `agency_code = USGS`.
3. Limitar inicialmente o retorno a cinco registros.
4. Validar a resposta HTTP.
5. Converter os registros recebidos em um DataFrame.
6. Inspecionar os identificadores, nomes, tipos de localização e geometrias.

### Conceitos técnicos
Parâmetros de consulta HTTP, filtros de API, `requests.get()`, `raise_for_status()` e `pd.json_normalize()`.

### Resultado esperado
Obter registros associados à agência USGS e verificar a disponibilidade das informações geográficas para as próximas etapas.


In [16]:

# Parâmetros da consulta
PARAMS_USGS = {
    "agency_code": "USGS",
    "limit": 5
}

# Requisição HTTP
response_usgs = requests.get(
    BASE_URL,
    params=PARAMS_USGS,
    timeout=30
)

# Validação da resposta
response_usgs.raise_for_status()

# Conversão para objeto Python
dados_usgs = response_usgs.json()

# Inspeção da resposta
print("Status HTTP:", response_usgs.status_code)
print("Tipo de retorno:", dados_usgs.get("type"))
print("Registros retornados:", dados_usgs.get("numberReturned"))

# Conversão para DataFrame
df_locais_usgs = pd.json_normalize(dados_usgs["features"])

# Seleção de campos para análise
colunas_usgs = [
    "id",
    "properties.agency_code",
    "properties.agency_name",
    "properties.monitoring_location_number",
    "properties.monitoring_location_name",
    "properties.site_type",
    "properties.state_name",
    "properties.county_name",
    "geometry"
]

# Inspeção das colunas disponíveis
print("Quantidade de colunas:", len(df_locais_usgs.columns))

print("\nColunas disponíveis:")
print(df_locais_usgs.columns.tolist())

# Visualização dos primeiros registros
display(df_locais_usgs.head())


Status HTTP: 200
Tipo de retorno: FeatureCollection
Registros retornados: 5
Quantidade de colunas: 47

Colunas disponíveis:
['type', 'id', 'properties.id', 'properties.agency_code', 'properties.agency_name', 'properties.monitoring_location_number', 'properties.monitoring_location_name', 'properties.district_code', 'properties.country_code', 'properties.country_name', 'properties.state_code', 'properties.state_name', 'properties.county_code', 'properties.county_name', 'properties.minor_civil_division_code', 'properties.site_type_code', 'properties.site_type', 'properties.hydrologic_unit_code', 'properties.basin_code', 'properties.altitude', 'properties.altitude_accuracy', 'properties.altitude_method_code', 'properties.altitude_method_name', 'properties.vertical_datum', 'properties.vertical_datum_name', 'properties.horizontal_positional_accuracy_code', 'properties.horizontal_positional_accuracy', 'properties.horizontal_position_method_code', 'properties.horizontal_position_method_name', 

,type,id,properties.id,properties.agency_code,properties.agency_name,properties.monitoring_location_number,properties.monitoring_location_name,properties.district_code,properties.country_code,properties.country_name,...,properties.national_aquifer_code,properties.aquifer_type_code,properties.well_constructed_depth,properties.hole_constructed_depth,properties.depth_source_code,properties.revision_note,properties.revision_created,properties.revision_modified,geometry.type,geometry.coordinates
0,Feature,USGS-00000000,USGS-00000000,USGS,U.S. Geological Survey,00000000,SAN MARCIAL ADDED,35,US,United States of America,...,None,None,None,None,None,None,None,None,Point,"[-106.995028, 33.677862]"
1,Feature,USGS-00000027,USGS-00000027,USGS,U.S. Geological Survey,00000027,DUMMY SITE - Testing Equipment (in house),36,US,United States of America,...,None,None,None,None,None,None,None,None,Point,"[-73.695556, 42.680556]"
2,Feature,USGS-00000028,USGS-00000028,USGS,U.S. Geological Survey,00000028,CERC LAB POCIS BLANK,36,US,United States of America,...,None,None,None,None,None,None,None,None,Point,"[-73.695556, 42.680556]"
3,Feature,USGS-00000029,USGS-00000029,USGS,NaN,00000029,DUMMY SITE - Coram Testing Equipment (in house),36,US,United States of America,...,None,None,None,None,None,None,None,None,Point,"[-73.007533, 40.873539]"
4,Feature,USGS-00004259,USGS-00004259,USGS,U.S. Geological Survey,00004259,E-10 Calumet,22,US,United States of America,...,None,None,None,None,None,None,None,None,Point,"[-91.374944, 29.696222]"



## 16. Identificação dos campos geográficos das localizações USGS

### Objetivo
Identificar quais campos da resposta da API armazenam informações geográficas das localizações monitoradas.

### Justificativa
A consulta filtrada pela agência USGS retornou 47 colunas, mas não apresentou o campo `geometry`. Precisamos identificar como as informações de latitude e longitude estão representadas nessa coleção.

### Estratégia
1. Inspecionar os nomes das colunas disponíveis.
2. Identificar campos relacionados a latitude, longitude e coordenadas.
3. Verificar os tipos de localização e os estados disponíveis.
4. Avaliar se os registros representam estações de monitoramento ou localizações de teste.

### Conceitos técnicos
Inspeção de esquema, manipulação de strings, filtros de colunas e análise exploratória com Pandas.

### Resultado esperado
Identificar os campos necessários para extrair as coordenadas geográficas e estabelecer critérios mais adequados para selecionar as estações.


In [17]:

# Identificação de colunas geográficas
termos_geograficos = [
    "latitude",
    "longitude",
    "coordinate",
    "geometry",
    "state",
    "county",
    "site_type"
]

colunas_geograficas = [
    coluna
    for coluna in df_locais_usgs.columns
    if any(
        termo in coluna.lower()
        for termo in termos_geograficos
    )
]

print("Campos identificados:")
for coluna in colunas_geograficas:
    print("-", coluna)

# Visualização dos campos encontrados
display(df_locais_usgs[colunas_geograficas])


Campos identificados:
- properties.state_code
- properties.state_name
- properties.county_code
- properties.county_name
- properties.site_type_code
- properties.site_type
- geometry.type
- geometry.coordinates


,properties.state_code,properties.state_name,properties.county_code,properties.county_name,properties.site_type_code,properties.site_type,geometry.type,geometry.coordinates
0,35,New Mexico,053,Socorro County,ST,Stream,Point,"[-106.995028, 33.677862]"
1,36,New York,083,Rensselaer County,FA-QC,Laboratory or sample-preparation area,Point,"[-73.695556, 42.680556]"
2,36,New York,083,Rensselaer County,FA-QC,Laboratory or sample-preparation area,Point,"[-73.695556, 42.680556]"
3,36,New York,103,Suffolk County,FA-QC,Laboratory or sample-preparation area,Point,"[-73.007533, 40.873539]"
4,22,Louisiana,101,St. Mary Parish,ST,Stream,Point,"[-91.374944, 29.696222]"



### Conclusão — Identificação dos campos geográficos

A inspeção do DataFrame identificou os campos relacionados à localização geográfica e à classificação das estações.

Foram identificados:

- `properties.state_name`: estado.
- `properties.county_name`: município ou condado.
- `properties.site_type_code`: código do tipo de localização.
- `properties.site_type`: descrição do tipo de localização.
- `geometry.type`: tipo de geometria.
- `geometry.coordinates`: coordenadas geográficas.

A análise demonstrou que as coordenadas estão disponíveis no formato GeoJSON, seguindo a ordem longitude e latitude.

Também foram identificados registros classificados como `ST` (Stream) e `FA-QC` (Laboratory or sample-preparation area).

**Resultado da etapa**

Foram identificados os campos necessários para selecionar localizações de cursos d'água e extrair suas coordenadas geográficas.

A próxima etapa será restringir os registros ao tipo `Stream`, excluindo localizações laboratoriais e outras categorias que não correspondem ao objetivo da análise.



## 17. Seleção de localizações de cursos d'água

### Objetivo
Selecionar as localizações classificadas como `Stream` e extrair suas coordenadas geográficas.

### Justificativa
A consulta anterior retornou diferentes tipos de localização. Para esta etapa do exercício, vamos priorizar os registros associados a cursos d'água, identificados pelo código `ST`.

### Estratégia
1. Filtrar os registros pelo código `ST`.
2. Criar uma cópia do DataFrame filtrado.
3. Extrair longitude e latitude da coluna `geometry.coordinates`.
4. Organizar os campos relevantes para análise.
5. Inspecionar os registros selecionados.

### Conceitos técnicos
Filtros com Pandas, `.loc[]`, `.copy()`, acesso a listas e criação de colunas.

### Resultado esperado
Obter um DataFrame com as localizações do tipo Stream e suas respectivas coordenadas geográficas.


In [18]:

# Seleção das localizações do tipo Stream
df_streams = df_locais_usgs.loc[
    df_locais_usgs["properties.site_type_code"] == "ST"
].copy()

# Extração das coordenadas geográficas
df_streams["longitude"] = df_streams["geometry.coordinates"].str[0]
df_streams["latitude"] = df_streams["geometry.coordinates"].str[1]

# Seleção dos campos relevantes
colunas_streams = [
    "properties.monitoring_location_number",
    "properties.monitoring_location_name",
    "properties.state_name",
    "properties.county_name",
    "properties.site_type",
    "longitude",
    "latitude"
]

# Inspeção dos resultados
print("Localizações do tipo Stream:", len(df_streams))

display(df_streams[colunas_streams])


Localizações do tipo Stream: 2


,properties.monitoring_location_number,properties.monitoring_location_name,properties.state_name,properties.county_name,properties.site_type,longitude,latitude
0,00000000,SAN MARCIAL ADDED,New Mexico,Socorro County,Stream,-106.995028,33.677862
4,00004259,E-10 Calumet,Louisiana,St. Mary Parish,Stream,-91.374944,29.696222



### Conclusão — Seleção de localizações do tipo Stream

O filtro aplicado ao campo `properties.site_type_code` identificou duas localizações classificadas como `ST` (Stream), entre os cinco registros retornados pela consulta anterior.

A extração das coordenadas geográficas foi realizada com sucesso, utilizando a coluna `geometry.coordinates`:

- Posição 0: longitude.
- Posição 1: latitude.

Foram criadas duas novas colunas no DataFrame:

- `longitude`
- `latitude`

**Resultado da etapa**

Foi criado o DataFrame `df_streams`, contendo as localizações classificadas como Stream e suas respectivas informações geográficas.

**Ponto de atenção**

Um dos registros apresenta o identificador `00000000` e o nome `SAN MARCIAL ADDED`. Esses dados precisam de validação antes de serem utilizados na análise.

A amostra atual está limitada a cinco registros. Portanto, ainda não representa o conjunto completo de localizações disponíveis na agência USGS.



## 18. Ampliação da amostra de localizações USGS

### Objetivo
Ampliar a quantidade de localizações consultadas para identificar mais registros classificados como Stream.

### Justificativa
A consulta anterior retornou somente cinco registros, dos quais dois foram classificados como Stream. Uma amostra maior permitirá avaliar melhor a disponibilidade de localizações e identificar possíveis registros inconsistentes.

### Estratégia
1. Aumentar o limite da consulta para 100 registros.
2. Manter o filtro da agência USGS.
3. Converter a resposta para DataFrame.
4. Identificar as localizações do tipo Stream.
5. Verificar a disponibilidade de coordenadas geográficas.

### Resultado esperado
Obter uma amostra maior de localizações USGS e avaliar a quantidade de registros Stream disponíveis.


In [19]:

# Ampliação dos parâmetros da consulta
PARAMS_USGS = {
    "agency_code": "USGS",
    "limit": 100
}

# Requisição HTTP
response_usgs = requests.get(
    BASE_URL,
    params=PARAMS_USGS,
    timeout=30
)

# Validação e conversão da resposta
response_usgs.raise_for_status()
dados_usgs = response_usgs.json()

# Conversão para DataFrame
df_locais_usgs = pd.json_normalize(dados_usgs["features"])

# Filtragem das localizações Stream
df_streams = df_locais_usgs.loc[
    df_locais_usgs["properties.site_type_code"] == "ST"
].copy()

# Extração das coordenadas
df_streams["longitude"] = df_streams["geometry.coordinates"].str[0]
df_streams["latitude"] = df_streams["geometry.coordinates"].str[1]

# Resumo da consulta
print("Status HTTP:", response_usgs.status_code)
print("Registros retornados:", len(df_locais_usgs))
print("Localizações Stream:", len(df_streams))
print(
    "Coordenadas ausentes:",
    df_streams[["longitude", "latitude"]].isna().any(axis=1).sum()
)

# Visualização
display(
    df_streams[
        [
            "properties.monitoring_location_number",
            "properties.monitoring_location_name",
            "properties.state_name",
            "properties.county_name",
            "longitude",
            "latitude"
        ]
    ].head(15)
)


Status HTTP: 200
Registros retornados: 100
Localizações Stream: 81
Coordenadas ausentes: 0


,properties.monitoring_location_number,properties.monitoring_location_name,properties.state_name,properties.county_name,longitude,latitude
0,00000000,SAN MARCIAL ADDED,New Mexico,Socorro County,-106.995028,33.677862
4,00004259,E-10 Calumet,Louisiana,St. Mary Parish,-91.374944,29.696222
5,00004261,E-08 Yellow Bayou (Gulf side),Louisiana,St. Mary Parish,-91.451944,29.740028
6,00004262,E-02 South of Krotz Springs,Louisiana,St. Landry Parish,-91.766306,30.460750
7,00004263,SSS-RT-005 Jean Lafitte,Louisiana,St. Martin Parish,-91.172111,29.769472
8,00004264,"Bayou Des Families on Hwy 3134 N Jean Lafitte, La",Louisiana,St. Landry Parish,-91.859083,30.549611
9,00004265,E-03 Charenton (Atchafalaya side),Louisiana,St. Mary Parish,-91.519778,29.901639
10,00004266,E-03 Charenton (Franklin side),Louisiana,St. Mary Parish,-91.519778,29.901639
11,00004267,Baro 1,Louisiana,St. Mary Parish,-91.522417,29.786944
12,00004268,"E-12 Bayou Choupique nr Baldwin, LA",Louisiana,St. Mary Parish,-91.567400,29.832639



### Conclusão — Ampliação da amostra de localizações USGS

A consulta foi ampliada para 100 registros, mantendo o filtro da agência USGS.

Após a aplicação do filtro para o código `ST` (Stream), foram identificadas 81 localizações classificadas como cursos d'água.

A extração das coordenadas geográficas foi concluída sem registros ausentes.

**Resultados obtidos**

- 100 localizações retornadas pela API.
- 81 localizações classificadas como Stream.
- 19 localizações pertencentes a outras categorias.
- Nenhuma coordenada ausente entre as localizações Stream selecionadas.

**Ponto de atenção**

A consulta utiliza um limite de 100 registros e não representa necessariamente o universo completo de localizações disponíveis na USGS.

Além disso, a classificação Stream, isoladamente, não garante que todos os registros sejam adequados para a análise ambiental.

**Resultado da etapa**

Foi obtida uma amostra ampliada de localizações USGS com coordenadas geográficas disponíveis, permitindo avançar para a avaliação da distribuição geográfica e da qualidade dos registros.



## 19. Análise da distribuição geográfica das localizações Stream

### Objetivo
Analisar a distribuição das localizações classificadas como Stream entre os estados americanos.

### Justificativa
A distribuição geográfica permite compreender a abrangência da amostra e identificar possíveis concentrações de registros em determinados estados.

### Estratégia
1. Agrupar as localizações por estado.
2. Contabilizar os registros de cada estado.
3. Calcular a participação percentual de cada estado na amostra.
4. Ordenar os resultados pela quantidade de localizações.

### Conceitos técnicos
Pandas GroupBy, agregações, contagem de registros, ordenação e cálculo percentual.

### Resultado esperado
Obter uma tabela com a quantidade e a participação percentual das localizações Stream por estado.


In [20]:
# Agrupamento das localizações por estado
df_estados = (
    df_streams
    .groupby("properties.state_name", dropna=False)
    .size()
    .reset_index(name="quantidade")
)

# Identificação dos estados ausentes
df_estados["properties.state_name"] = (
    df_estados["properties.state_name"].fillna("Não informado")
)

# Cálculo da participação percentual
df_estados["participacao_percentual"] = (
    df_estados["quantidade"] / len(df_streams) * 100
).round(2)

# Ordenação decrescente
df_estados = df_estados.sort_values(
    by="quantidade",
    ascending=False
)

# Visualização
display(df_estados)

print("Quantidade de categorias:", len(df_estados))
print("Total de localizações:", df_estados["quantidade"].sum())

,properties.state_name,quantidade,participacao_percentual
1,Maine,67,82.72
0,Louisiana,12,14.81
2,New Mexico,1,1.23
3,Não informado,1,1.23


Quantidade de categorias: 4
Total de localizações: 81



### Conclusão: Distribuição geográfica das localizações Stream

A análise da distribuição geográfica identificou 81 localizações classificadas como Stream, distribuídas em três estados americanos com informações disponíveis.

O estado do Maine concentra 67 registros (82,72%), seguido pela Louisiana, com 12 registros (14,81%), e pelo New Mexico, com um registro (1,23%).

Durante a conferência dos totais, identificamos uma localização sem o estado informado. Trata-se do registro `01011500`, denominado St. Francis River near Connors, New Brunswick.

Essa ausência explica a diferença entre as 81 localizações identificadas e as 80 contabilizadas no agrupamento por estado.

**Resultado da etapa**

A distribuição geográfica foi caracterizada e a inconsistência encontrada foi identificada. O registro sem estado será preservado na base, evitando a perda de informação durante as próximas transformações.



## 20. Verificação de informações geográficas ausentes

### Objetivo
Investigar a diferença entre a quantidade de localizações Stream identificadas e o total contabilizado no agrupamento por estado.

### Justificativa
A análise anterior identificou 81 localizações Stream, mas o agrupamento por estado contabilizou 80 registros. Como o Pandas exclui valores ausentes durante o agrupamento padrão, precisamos verificar se existe alguma localização sem estado informado.

### Estratégia
1. Contabilizar os valores ausentes no campo de estado.
2. Identificar os registros que não possuem essa informação.
3. Comparar a quantidade total com os registros agrupados.

### Resultado esperado
Confirmar a origem da diferença e avaliar a completude do campo geográfico.


In [21]:

# Verificação de estados ausentes
estados_ausentes = df_streams[
    df_streams["properties.state_name"].isna()
]

print(
    "Localizações sem estado informado:",
    len(estados_ausentes)
)

# Exibição dos registros identificados
display(estados_ausentes)

# Conferência dos totais
print("Total de localizações Stream:", len(df_streams))
print("Total contabilizado por estado:", df_estados["quantidade"].sum())


Localizações sem estado informado: 1


,type,id,properties.id,properties.agency_code,properties.agency_name,properties.monitoring_location_number,properties.monitoring_location_name,properties.district_code,properties.country_code,properties.country_name,...,properties.well_constructed_depth,properties.hole_constructed_depth,properties.depth_source_code,properties.revision_note,properties.revision_created,properties.revision_modified,geometry.type,geometry.coordinates,longitude,latitude
62,Feature,USGS-01011500,USGS-01011500,USGS,U.S. Geological Survey,01011500,"St. Francis River near Connors, New Brunswick",23,CA,Canada,...,None,None,None,WDR ME-82-1: Drainage area. WDR ME-97-1: 1992(...,2017-11-13T06:00:00+00:00,NaN,Point,"[-68.956944, 47.206944]",-68.956944,47.206944


Total de localizações Stream: 81
Total contabilizado por estado: 81



### Conclusão: Distribuição geográfica das localizações Stream

A análise identificou 81 localizações classificadas como Stream, distribuídas entre três estados americanos e uma categoria sem estado informado.

O Maine concentra a maior parte dos registros, com 67 localizações (82,72%). A Louisiana possui 12 registros (14,81%), enquanto o New Mexico possui uma localização (1,23%).

Também foi identificada uma localização sem estado informado. Para preservar esse registro, ajustamos o agrupamento utilizando `dropna=False` e substituímos o valor ausente por "Não informado" com `fillna()`.

**Resultado da etapa**

A distribuição passou a contabilizar os 81 registros, sem perda de informações. A análise também demonstrou a importância de tratar valores ausentes durante as transformações com Pandas.



## 21. Validação dos identificadores e nomes das localizações

### Objetivo
Avaliar a qualidade dos identificadores e nomes das localizações classificadas como Stream.

### Justificativa
A análise anterior identificou registros com características que merecem atenção, como o identificador `00000000` e o nome `SAN MARCIAL ADDED`.

Antes de utilizar essas localizações em outras análises, precisamos verificar a existência de identificadores ausentes, duplicados e nomes que possam indicar registros de teste ou inconsistências cadastrais.

### Estratégia
1. Verificar identificadores ausentes ou vazios.
2. Identificar possíveis identificadores duplicados.
3. Verificar nomes ausentes ou vazios.
4. Procurar termos que indiquem registros de teste ou nomenclaturas incomuns.
5. Exibir os registros que precisam de uma análise adicional.

### Conceitos técnicos
Pandas, `isna()`, `duplicated()`, `str.contains()` e expressões regulares.


In [22]:

# Campos que serão avaliados
coluna_id = "properties.monitoring_location_number"
coluna_nome = "properties.monitoring_location_name"

# Verificação de identificadores ausentes ou vazios
ids_ausentes = (
    df_streams[coluna_id].isna()
    | df_streams[coluna_id].astype(str).str.strip().eq("")
)

# Identificação de identificadores duplicados
ids_duplicados = df_streams[coluna_id].duplicated(keep=False)

# Verificação de nomes ausentes ou vazios
nomes_ausentes = (
    df_streams[coluna_nome].isna()
    | df_streams[coluna_nome].astype(str).str.strip().eq("")
)

# Busca de termos que merecem investigação
termos_suspeitos = r"DUMMY|TEST|ADDED|BLANK"

nomes_suspeitos = df_streams[coluna_nome].str.contains(
    termos_suspeitos,
    case=False,
    na=False,
    regex=True
)

# Resumo da qualidade dos registros
print("Total de localizações:", len(df_streams))
print("Identificadores ausentes ou vazios:", ids_ausentes.sum())
print("Registros com identificadores duplicados:", ids_duplicados.sum())
print("Nomes ausentes ou vazios:", nomes_ausentes.sum())
print("Nomes com termos suspeitos:", nomes_suspeitos.sum())

# Exibição dos registros que merecem investigação
df_validacao = df_streams.loc[
    ids_ausentes | ids_duplicados | nomes_ausentes | nomes_suspeitos,
    [
        coluna_id,
        coluna_nome,
        "properties.state_name",
        "properties.county_name",
        "longitude",
        "latitude"
    ]
].copy()

display(df_validacao)


Total de localizações: 81
Identificadores ausentes ou vazios: 0
Registros com identificadores duplicados: 0
Nomes ausentes ou vazios: 0
Nomes com termos suspeitos: 1


,properties.monitoring_location_number,properties.monitoring_location_name,properties.state_name,properties.county_name,longitude,latitude
0,00000000,SAN MARCIAL ADDED,New Mexico,Socorro County,-106.995028,33.677862



### Conclusão: Validação dos identificadores e nomes

A verificação dos 81 registros identificou que todos possuem identificadores e nomes preenchidos, sem ocorrências de identificadores duplicados.

Durante a análise, foi encontrado um registro com nomenclatura que merece atenção: `SAN MARCIAL ADDED`, associado ao identificador `00000000`.

Embora esse registro tenha sido sinalizado pela busca de termos suspeitos, não existem evidências suficientes para classificá-lo como inválido.

**Resultado da etapa**

A amostra não apresentou problemas de preenchimento ou duplicidade nos campos avaliados. Uma localização foi sinalizada para investigação adicional e será mantida na base até que sua validade seja confirmada.



## 22. Preparação da base de localizações

### Objetivo
Criar uma estrutura organizada com os principais atributos das localizações classificadas como Stream.

### Justificativa
Após as verificações iniciais, precisamos consolidar os campos relevantes em uma estrutura tabular mais simples, facilitando a utilização dos dados nas próximas etapas.

### Estratégia
1. Selecionar os identificadores e nomes das localizações.
2. Manter as informações de estado e condado.
3. Preservar o tipo de localização e as coordenadas geográficas.
4. Criar uma cópia independente do DataFrame original.
5. Conferir a estrutura e a quantidade de registros.

### Resultado esperado
Obter uma base organizada de localizações USGS, mantendo os 81 registros analisados.


In [23]:

# Seleção e organização dos campos
df_localizacoes = df_streams[
    [
        "properties.monitoring_location_number",
        "properties.monitoring_location_name",
        "properties.state_name",
        "properties.county_name",
        "properties.site_type",
        "longitude",
        "latitude"
    ]
].copy()

# Renomeação das colunas
df_localizacoes.columns = [
    "codigo_localizacao",
    "nome_localizacao",
    "estado",
    "condado",
    "tipo_localizacao",
    "longitude",
    "latitude"
]

# Verificação da estrutura
print("Total de registros:", len(df_localizacoes))
print("Total de colunas:", len(df_localizacoes.columns))

display(df_localizacoes.head(10))

Total de registros: 81
Total de colunas: 7


,codigo_localizacao,nome_localizacao,estado,condado,tipo_localizacao,longitude,latitude
0,00000000,SAN MARCIAL ADDED,New Mexico,Socorro County,Stream,-106.995028,33.677862
4,00004259,E-10 Calumet,Louisiana,St. Mary Parish,Stream,-91.374944,29.696222
5,00004261,E-08 Yellow Bayou (Gulf side),Louisiana,St. Mary Parish,Stream,-91.451944,29.740028
6,00004262,E-02 South of Krotz Springs,Louisiana,St. Landry Parish,Stream,-91.766306,30.460750
7,00004263,SSS-RT-005 Jean Lafitte,Louisiana,St. Martin Parish,Stream,-91.172111,29.769472
8,00004264,"Bayou Des Families on Hwy 3134 N Jean Lafitte, La",Louisiana,St. Landry Parish,Stream,-91.859083,30.549611
9,00004265,E-03 Charenton (Atchafalaya side),Louisiana,St. Mary Parish,Stream,-91.519778,29.901639
10,00004266,E-03 Charenton (Franklin side),Louisiana,St. Mary Parish,Stream,-91.519778,29.901639
11,00004267,Baro 1,Louisiana,St. Mary Parish,Stream,-91.522417,29.786944
12,00004268,"E-12 Bayou Choupique nr Baldwin, LA",Louisiana,St. Mary Parish,Stream,-91.567400,29.832639



### Conclusão: Preparação da base de localizações

Após as etapas de exploração e validação, organizamos os dados das localizações Stream em um novo DataFrame chamado `df_localizacoes`.

A estrutura final contém 81 registros e sete colunas: código, nome, estado, condado, tipo de localização, longitude e latitude.

Os nomes das colunas foram simplificados para facilitar a leitura e a utilização dos dados nas próximas etapas.

Os identificadores foram preservados, inclusive aqueles que possuem zeros à esquerda. Também mantivemos o registro que apresenta o estado não informado e a localização sinalizada anteriormente, sem realizar exclusões automáticas.

**Resultado da etapa**

A base de localizações está organizada e pronta para as próximas análises, mantendo os registros e os atributos geográficos identificados durante a exploração da API USGS.



## 23. Validação das coordenadas geográficas

### Objetivo
Verificar a consistência das coordenadas de latitude e longitude das localizações USGS.

### Justificativa
As coordenadas geográficas serão utilizadas posteriormente para consultar informações meteorológicas no Open-Meteo. Por isso, precisamos verificar se os valores estão preenchidos, dentro dos intervalos permitidos e se existem coordenadas repetidas.

### Estratégia
1. Verificar valores ausentes nas colunas de latitude e longitude.
2. Validar os intervalos geográficos permitidos.
3. Identificar coordenadas duplicadas.
4. Contabilizar os registros que atendem aos critérios de validação.

### Conceitos técnicos
Pandas, `isna()`, operadores lógicos, filtros booleanos, `between()` e `duplicated()`.

### Resultado esperado
Identificar possíveis inconsistências nas coordenadas e avaliar a qualidade geográfica da base de localizações.


In [24]:

# Verificação de coordenadas ausentes
coordenadas_ausentes = (
    df_localizacoes["longitude"].isna()
    | df_localizacoes["latitude"].isna()
)

# Validação dos intervalos geográficos
longitude_valida = df_localizacoes["longitude"].between(-180, 180)
latitude_valida = df_localizacoes["latitude"].between(-90, 90)

# Identificação de coordenadas fora dos intervalos permitidos
coordenadas_invalidas = ~(longitude_valida & latitude_valida)

# Identificação de coordenadas duplicadas
coordenadas_duplicadas = df_localizacoes.duplicated(
    subset=["longitude", "latitude"],
    keep=False
)

# Resumo da validação
print("Total de localizações:", len(df_localizacoes))
print("Coordenadas ausentes:", coordenadas_ausentes.sum())
print("Coordenadas fora do intervalo:", coordenadas_invalidas.sum())
print("Registros com coordenadas duplicadas:", coordenadas_duplicadas.sum())

# Exibição dos registros com coordenadas duplicadas
df_coordenadas_duplicadas = df_localizacoes.loc[
    coordenadas_duplicadas
].sort_values(
    by=["longitude", "latitude"]
)

display(df_coordenadas_duplicadas)


Total de localizações: 81
Coordenadas ausentes: 0
Coordenadas fora do intervalo: 0
Registros com coordenadas duplicadas: 32


,codigo_localizacao,nome_localizacao,estado,condado,tipo_localizacao,longitude,latitude
9,00004265,E-03 Charenton (Atchafalaya side),Louisiana,St. Mary Parish,Stream,-91.519778,29.901639
10,00004266,E-03 Charenton (Franklin side),Louisiana,St. Mary Parish,Stream,-91.519778,29.901639
38,0101050001,ST JOHN R A DICKEY ME 01,Maine,Aroostook County,Stream,-69.089762,47.112257
39,0101050002,ST JOHN R A DICKEY ME 02,Maine,Aroostook County,Stream,-69.089762,47.112257
40,0101050003,ST JOHN R A DICKEY ME 03,Maine,Aroostook County,Stream,-69.089762,47.112257
41,0101050004,ST JOHN R A DICKEY ME 04,Maine,Aroostook County,Stream,-69.089762,47.112257
42,0101050005,ST JOHN R A DICKEY ME 05,Maine,Aroostook County,Stream,-69.089762,47.112257
43,0101050006,ST JOHN R A DICKEY ME 06,Maine,Aroostook County,Stream,-69.089762,47.112257
44,0101050007,ST JOHN R A DICKEY ME 07,Maine,Aroostook County,Stream,-69.089762,47.112257
45,0101050008,ST JOHN R A DICKEY ME 08,Maine,Aroostook County,Stream,-69.089762,47.112257



### Conclusão: Validação das coordenadas geográficas

A validação das coordenadas identificou que as 81 localizações possuem latitude e longitude preenchidas, com todos os valores dentro dos intervalos geográficos permitidos.

Durante a análise, foram encontrados 32 registros com coordenadas duplicadas. Ao observar os exemplos, identificamos diferentes localizações que compartilham o mesmo ponto geográfico, como os registros associados ao rio St. John e ao rio Allagash, no estado do Maine.

Essa situação não representa necessariamente um problema, pois diferentes pontos de monitoramento podem estar associados à mesma coordenada geográfica.

**Resultado da etapa**

As coordenadas foram consideradas válidas quanto ao preenchimento e aos intervalos geográficos. Os registros com coordenadas compartilhadas serão preservados, pois representam localizações distintas e não devem ser eliminados automaticamente.



## 24. Análise das coordenadas geográficas compartilhadas

### Objetivo
Identificar a quantidade de pontos geográficos distintos e analisar quantas localizações estão associadas às mesmas coordenadas.

### Justificativa
A validação anterior identificou registros com coordenadas duplicadas. Precisamos compreender melhor essa distribuição para evitar a eliminação indevida de localizações e avaliar como os dados meteorológicos poderão ser associados posteriormente.

### Estratégia
1. Agrupar os registros por longitude e latitude.
2. Contabilizar as localizações associadas a cada ponto.
3. Identificar os pontos compartilhados por duas ou mais localizações.
4. Calcular a quantidade de pontos geográficos distintos.

### Resultado esperado
Conhecer a distribuição das localizações por ponto geográfico e dimensionar a quantidade de coordenadas únicas disponíveis.


In [25]:

# Agrupamento das localizações por coordenadas
df_pontos_geograficos = (
    df_localizacoes
    .groupby(["longitude", "latitude"])
    .agg(
        quantidade_localizacoes=("codigo_localizacao", "count"),
        nomes_localizacoes=("nome_localizacao", lambda x: list(x))
    )
    .reset_index()
)

# Identificação dos pontos compartilhados
df_pontos_compartilhados = df_pontos_geograficos.loc[
    df_pontos_geograficos["quantidade_localizacoes"] > 1
].copy()

# Resumo da análise
print("Total de localizações:", len(df_localizacoes))
print("Pontos geográficos distintos:", len(df_pontos_geograficos))
print("Pontos compartilhados:", len(df_pontos_compartilhados))
print(
    "Localizações em pontos compartilhados:",
    df_pontos_compartilhados["quantidade_localizacoes"].sum()
)

# Visualização dos pontos compartilhados
display(
    df_pontos_compartilhados.sort_values(
        by="quantidade_localizacoes",
        ascending=False
    )
)


Total de localizações: 81
Pontos geográficos distintos: 53
Pontos compartilhados: 4
Localizações em pontos compartilhados: 32


,longitude,latitude,quantidade_localizacoes,nomes_localizacoes
20,-69.089762,47.112257,10,"[ST JOHN R A DICKEY ME 01, ST JOHN R A DICKEY ..."
22,-69.080318,47.070591,10,"[ALLAGASH R NR ALLAGASH ME 01, ALLAGASH R NR A..."
37,-68.592547,47.257539,10,"[ST JOHN R BLW FISH R A FT KENT ME 01, ST JOHN..."
6,-91.519778,29.901639,2,"[E-03 Charenton (Atchafalaya side), E-03 Chare..."



### Conclusão: Análise das coordenadas geográficas compartilhadas

A análise identificou 53 pontos geográficos distintos entre as 81 localizações USGS avaliadas.

Foram encontrados quatro pontos compartilhados por diferentes localizações, totalizando 32 registros. Os demais 49 pontos possuem uma única localização associada.

Essa característica demonstra que diferentes identificadores de monitoramento podem representar localizações cadastradas nas mesmas coordenadas geográficas.

**Resultado da etapa**

A análise permitiu distinguir as 81 localizações cadastradas dos 53 pontos geográficos disponíveis.

Essa distinção será considerada na integração com o Open-Meteo, permitindo consultar cada par de coordenadas uma única vez e posteriormente relacionar os dados meteorológicos às localizações correspondentes, sem eliminar registros da USGS.



## 25. Preparação dos pontos geográficos únicos

### Objetivo
Criar uma base contendo apenas os pontos geográficos distintos identificados nas localizações USGS.

### Justificativa
A análise anterior demonstrou que diferentes localizações podem compartilhar as mesmas coordenadas. Para evitar consultas meteorológicas repetidas, precisamos organizar os pontos geográficos de forma independente dos registros de monitoramento.

### Estratégia
1. Utilizar o agrupamento geográfico realizado anteriormente.
2. Criar um identificador para cada ponto geográfico.
3. Manter longitude e latitude.
4. Preservar a quantidade de localizações associadas a cada ponto.
5. Conferir a quantidade de pontos gerados.

### Resultado esperado
Obter um DataFrame com 53 pontos geográficos únicos, preparado para as futuras consultas ao Open-Meteo.


In [26]:

# Criação da base de pontos geográficos únicos
df_pontos_unicos = (
    df_pontos_geograficos[
        [
            "longitude",
            "latitude",
            "quantidade_localizacoes"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

# Criação de identificador para cada ponto
df_pontos_unicos.insert(
    0,
    "id_ponto",
    range(1, len(df_pontos_unicos) + 1)
)

# Resumo da preparação
print("Pontos geográficos únicos:", len(df_pontos_unicos))
print(
    "Localizações representadas:",
    df_pontos_unicos["quantidade_localizacoes"].sum()
)

# Visualização
display(df_pontos_unicos.head(10))


Pontos geográficos únicos: 53
Localizações representadas: 81


,id_ponto,longitude,latitude,quantidade_localizacoes
0,1,-106.995028,33.677862,1
1,2,-91.859083,30.549611,1
2,3,-91.766306,30.460750,1
3,4,-91.567400,29.832639,1
4,5,-91.528000,29.785231,1
5,6,-91.522417,29.786944,1
6,7,-91.519778,29.901639,2
7,8,-91.496056,29.754028,1
8,9,-91.451944,29.740028,1
9,10,-91.374944,29.696222,1



### Conclusão: Preparação dos pontos geográficos únicos

A partir das 81 localizações USGS, criamos um novo DataFrame chamado `df_pontos_unicos`, contendo 53 pares distintos de longitude e latitude.

Cada ponto recebeu um identificador sequencial (`id_ponto`) e manteve a quantidade de localizações associadas às suas coordenadas.

A conferência dos resultados confirmou que os 53 pontos representam corretamente as 81 localizações originais.

**Resultado da etapa**

A base de pontos geográficos únicos está preparada para a integração com o Open-Meteo. Essa organização permitirá evitar consultas meteorológicas repetidas para coordenadas compartilhadas, preservando a relação com as localizações originais da USGS.



## 26. Primeira consulta à API Open-Meteo

### Objetivo
Realizar a primeira consulta à API Open-Meteo utilizando as coordenadas de um ponto geográfico identificado na base USGS.

### Justificativa
Após organizar as 81 localizações em 53 pontos geográficos únicos, podemos iniciar a exploração dos dados meteorológicos.

A primeira consulta será realizada individualmente para validar a comunicação com a API e compreender a estrutura da resposta antes de ampliar o processamento para os demais pontos.

### Estratégia
1. Selecionar o primeiro ponto de `df_pontos_unicos`.
2. Obter suas coordenadas de latitude e longitude.
3. Configurar as variáveis meteorológicas desejadas.
4. Executar a requisição HTTP.
5. Validar a resposta recebida.
6. Inspecionar os metadados e os dados horários.
7. Converter os dados meteorológicos em um DataFrame.

### Conceitos técnicos
API REST, parâmetros HTTP, JSON, `requests.get()`, `raise_for_status()` e `pd.DataFrame()`.

### Resultado esperado
Obter os dados meteorológicos horários de um ponto geográfico e compreender a estrutura retornada pela API Open-Meteo.


In [27]:

# Endpoint da API Open-Meteo
BASE_URL_METEO = "https://api.open-meteo.com/v1/forecast"

# Seleção do primeiro ponto geográfico
ponto = df_pontos_unicos.iloc[0]

# Parâmetros da consulta
PARAMS_METEO = {
    "latitude": ponto["latitude"],
    "longitude": ponto["longitude"],
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "wind_speed_10m"
    ],
    "forecast_days": 1,
    "timezone": "auto"
}

# Requisição HTTP
response_meteo = requests.get(
    BASE_URL_METEO,
    params=PARAMS_METEO,
    timeout=30
)

# Validação da resposta
response_meteo.raise_for_status()

# Conversão para objeto Python
dados_meteo = response_meteo.json()

# Inspeção da resposta
print("Status HTTP:", response_meteo.status_code)
print("Latitude solicitada:", ponto["latitude"])
print("Longitude solicitada:", ponto["longitude"])
print("Latitude retornada:", dados_meteo.get("latitude"))
print("Longitude retornada:", dados_meteo.get("longitude"))
print("Fuso horário:", dados_meteo.get("timezone"))
print("Altitude:", dados_meteo.get("elevation"))

# Conversão dos dados horários para DataFrame
df_meteo = pd.DataFrame(dados_meteo["hourly"])

# Inspeção do resultado
print("\nQuantidade de registros:", len(df_meteo))
print("Colunas disponíveis:", df_meteo.columns.tolist())

display(df_meteo.head(10))


Status HTTP: 200
Latitude solicitada: 33.67786220285366
Longitude solicitada: -106.9950280968082
Latitude retornada: 33.670307
Longitude retornada: -106.988144
Fuso horário: America/Denver
Altitude: 1377.0

Quantidade de registros: 24
Colunas disponíveis: ['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m']


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m
0,2026-11-19T00:00,5.9,72,0.0,2.8
1,2026-11-19T01:00,5.3,72,0.0,2.3
2,2026-11-19T02:00,4.7,70,0.0,2.9
3,2026-11-19T03:00,4.0,76,0.0,4.6
4,2026-11-19T04:00,3.7,80,0.0,6.3
5,2026-11-19T05:00,3.3,81,0.0,6.2
6,2026-11-19T06:00,2.6,84,0.0,3.7
7,2026-11-19T07:00,2.3,81,0.0,3.5
8,2026-11-19T08:00,4.9,85,0.0,6.1
9,2026-11-19T09:00,7.9,69,0.0,7.4



### Conclusão: Primeira consulta à API Open-Meteo

A primeira requisição à API Open-Meteo foi executada com sucesso, retornando o status HTTP 200.

A consulta utilizou as coordenadas do primeiro ponto geográfico disponível em `df_pontos_unicos` e retornou 24 registros horários, contendo temperatura, umidade relativa, precipitação e velocidade do vento.

Também foram identificados os metadados da localização utilizada pelo modelo meteorológico, incluindo latitude, longitude, altitude e fuso horário.

Durante a comparação das coordenadas, observamos pequenas diferenças entre o ponto solicitado e a localização retornada pela API. Isso ocorre porque o Open-Meteo utiliza a célula do modelo meteorológico mais próxima da coordenada informada.

**Resultado da etapa**

A comunicação com a API foi validada e os dados meteorológicos foram convertidos em um DataFrame chamado `df_meteo`.

A estrutura da resposta está compreendida e poderá servir de referência para ampliar as consultas aos demais pontos geográficos.



## 27. Diagnóstico dos dados meteorológicos

### Objetivo
Avaliar a estrutura, os tipos de dados e a presença de valores ausentes no DataFrame meteorológico.

### Justificativa
Antes de ampliar o processamento para os demais pontos geográficos, precisamos verificar a qualidade dos dados recebidos e identificar possíveis inconsistências.

### Estratégia
1. Inspecionar os tipos de dados das colunas.
2. Verificar a quantidade de valores ausentes.
3. Identificar registros duplicados.
4. Conferir a quantidade de registros e colunas.

### Conceitos técnicos
Pandas, `info()`, `isna()`, `sum()`, `duplicated()` e `shape`.


In [28]:

# Informações estruturais do DataFrame
print("Dimensões:", df_meteo.shape)

print("\nTipos de dados:")
print(df_meteo.dtypes)

# Verificação de valores ausentes
print("\nValores ausentes por coluna:")
print(df_meteo.isna().sum())

# Verificação de registros duplicados
print("\nRegistros duplicados:", df_meteo.duplicated().sum())

# Informações gerais
print("\nEstrutura do DataFrame:")
df_meteo.info()


Dimensões: (24, 5)

Tipos de dados:
time                        str
temperature_2m          float64
relative_humidity_2m      int64
precipitation           float64
wind_speed_10m          float64
dtype: object

Valores ausentes por coluna:
time                    0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
wind_speed_10m          0
dtype: int64

Registros duplicados: 0

Estrutura do DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   time                  24 non-null     str    
 1   temperature_2m        24 non-null     float64
 2   relative_humidity_2m  24 non-null     int64  
 3   precipitation         24 non-null     float64
 4   wind_speed_10m        24 non-null     float64
dtypes: float64(3), int64(1), str(1)
memory usage: 1.1 KB



### Conclusão: Diagnóstico dos dados meteorológicos

A inspeção do DataFrame `df_meteo` confirmou a presença de 24 registros e cinco colunas, correspondentes ao horário e às quatro variáveis meteorológicas solicitadas.

Não foram identificados valores ausentes ou registros duplicados. As variáveis numéricas também apresentaram tipos de dados adequados para as próximas análises.

Foi identificado que a coluna `time` está armazenada como texto. Essa característica será tratada na próxima etapa, permitindo trabalhar com os registros temporais utilizando os recursos do Pandas.

**Resultado da etapa**

A estrutura dos dados meteorológicos foi validada. Não foram encontradas inconsistências de preenchimento ou duplicidade na amostra analisada.



## 28. Conversão e validação dos dados temporais

### Objetivo
Converter a coluna `time` para o tipo datetime e verificar o intervalo temporal dos registros meteorológicos.

### Justificativa
Os dados horários retornados pela API estão armazenados inicialmente como texto. A conversão para datetime permite realizar operações temporais, ordenar registros cronologicamente e futuramente relacionar diferentes períodos de observação.

### Estratégia
1. Converter a coluna `time` utilizando `pd.to_datetime()`.
2. Verificar o tipo de dado resultante.
3. Identificar a primeira e a última data e hora disponíveis.
4. Conferir a quantidade de registros após a conversão.

### Conceitos técnicos
Pandas, datetime, `pd.to_datetime()`, `dtype`, `min()` e `max()`.

### Resultado esperado
Obter uma coluna temporal estruturada e validar o intervalo dos 24 registros meteorológicos.


In [29]:

# Conversão da coluna temporal
df_meteo["time"] = pd.to_datetime(df_meteo["time"])

# Validação do tipo de dado
print("Tipo da coluna time:", df_meteo["time"].dtype)

# Identificação do intervalo temporal
print("Primeiro registro:", df_meteo["time"].min())
print("Último registro:", df_meteo["time"].max())

# Conferência da quantidade de registros
print("Total de registros:", len(df_meteo))

# Visualização dos dados
display(df_meteo.head())


Tipo da coluna time: datetime64[us]
Primeiro registro: 2026-11-19 00:00:00
Último registro: 2026-11-19 23:00:00
Total de registros: 24


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m
0,2026-11-19 00:00:00,5.9,72,0.0,2.8
1,2026-11-19 01:00:00,5.3,72,0.0,2.3
2,2026-11-19 02:00:00,4.7,70,0.0,2.9
3,2026-11-19 03:00:00,4.0,76,0.0,4.6
4,2026-11-19 04:00:00,3.7,80,0.0,6.3



### Conclusão: Conversão e validação dos dados temporais

A coluna `time` foi convertida com sucesso para o tipo `datetime64[us]`, permitindo que o Pandas reconheça os valores como datas e horários.

A validação confirmou a presença de 24 registros, abrangendo o período de 00:00 até 23:00 do dia 19 de novembro de 2026.

Os horários correspondem ao fuso `America/Denver`, identificado na resposta da API Open-Meteo.

**Resultado da etapa**

A estrutura temporal foi preparada para as próximas transformações. Os registros agora podem ser utilizados em operações de ordenação, agrupamento e análise de séries temporais.



## 29. Criação de função para consulta meteorológica

### Objetivo
Criar uma função reutilizável para consultar os dados meteorológicos de qualquer ponto geográfico.

### Justificativa
A primeira consulta ao Open-Meteo foi realizada individualmente e validou a estrutura da resposta. Para ampliar o processamento aos 53 pontos, precisamos organizar a lógica da requisição em uma função.

### Estratégia
1. Definir uma função que receba latitude e longitude.
2. Configurar as variáveis meteorológicas desejadas.
3. Executar a requisição à API Open-Meteo.
4. Validar a resposta HTTP.
5. Organizar os dados horários em um DataFrame.
6. Preservar as coordenadas e os metadados retornados pela API.

### Conceitos técnicos
Python, funções, parâmetros, retorno de valores, Requests e Pandas.

### Resultado esperado
Disponibilizar uma função que possa ser reutilizada nas consultas dos 53 pontos geográficos.


In [31]:

# Função para consultar dados meteorológicos
def consultar_meteorologia(latitude, longitude):

    # Parâmetros da consulta
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "wind_speed_10m"
        ],
        "forecast_days": 1,
        "timezone": "auto"
    }

    # Requisição à API
    response = requests.get(
        BASE_URL_METEO,
        params=params,
        timeout=30
    )

    # Validação da resposta
    response.raise_for_status()

    # Conversão da resposta JSON
    dados = response.json()

    # Conversão dos dados horários em DataFrame
    df = pd.DataFrame(dados["hourly"])

    # Inclusão dos metadados geográficos
    df["latitude_solicitada"] = latitude
    df["longitude_solicitada"] = longitude
    df["latitude_retorno"] = dados["latitude"]
    df["longitude_retorno"] = dados["longitude"]
    df["timezone"] = dados["timezone"]
    df["elevation"] = dados["elevation"]

    # Conversão temporal
    df["time"] = pd.to_datetime(df["time"])

    return df


# Teste da função com o primeiro ponto geográfico
teste_meteo = consultar_meteorologia(
    ponto["latitude"],
    ponto["longitude"]
)

# Inspeção do resultado
print("Registros retornados:", len(teste_meteo))
print("Colunas disponíveis:", teste_meteo.columns.tolist())

display(teste_meteo.head())


Registros retornados: 24
Colunas disponíveis: ['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'latitude_solicitada', 'longitude_solicitada', 'latitude_retorno', 'longitude_retorno', 'timezone', 'elevation']


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,latitude_solicitada,longitude_solicitada,latitude_retorno,longitude_retorno,timezone,elevation
0,2026-11-19 00:00:00,5.9,72,0.0,2.8,33.677862,-106.995028,33.670307,-106.988144,America/Denver,1377.0
1,2026-11-19 01:00:00,5.3,72,0.0,2.3,33.677862,-106.995028,33.670307,-106.988144,America/Denver,1377.0
2,2026-11-19 02:00:00,4.7,70,0.0,2.9,33.677862,-106.995028,33.670307,-106.988144,America/Denver,1377.0
3,2026-11-19 03:00:00,4.0,76,0.0,4.6,33.677862,-106.995028,33.670307,-106.988144,America/Denver,1377.0
4,2026-11-19 04:00:00,3.7,80,0.0,6.3,33.677862,-106.995028,33.670307,-106.988144,America/Denver,1377.0



### Conclusão: Criação da função para consulta meteorológica

A função `consultar_meteorologia()` foi criada e testada com sucesso utilizando o primeiro ponto geográfico da base `df_pontos_unicos`.

A execução retornou 24 registros horários e manteve as quatro variáveis meteorológicas solicitadas. Também foram incorporados os metadados da consulta, incluindo as coordenadas informadas, as coordenadas retornadas pela API, o fuso horário e a altitude.

A comparação com a consulta individual anterior confirmou que a função reproduz a estrutura esperada dos dados.

**Resultado da etapa**

A lógica de consulta meteorológica foi organizada em uma função reutilizável, permitindo aplicar o mesmo processamento aos demais pontos geográficos sem repetir o código.



## 30. Conclusão da etapa de Data Discovery

### Resultado

A exploração das fontes USGS e Open-Meteo permitiu identificar as estruturas dos dados, os parâmetros ambientais disponíveis e as informações geográficas das localizações de monitoramento.

Na USGS, foram exploradas 11.946 características e identificadas 81 localizações classificadas como Stream, representadas por 53 pontos geográficos únicos.

Na Open-Meteo, foi realizada uma consulta de validação utilizando um desses pontos, com retorno de 24 registros horários e quatro variáveis meteorológicas.

Também foram realizados testes de qualidade, conversão temporal e criação de uma função reutilizável para consultas meteorológicas.

### Aprendizados

Foram aplicados conceitos de consumo de APIs REST, tratamento de respostas JSON, manipulação de dados com Pandas, análise geográfica, validação de dados e criação de funções em Python.

### Próxima etapa

Com a estrutura inicial das fontes conhecida, o desenvolvimento seguirá para a implementação do pipeline de dados, priorizando os requisitos técnicos e os entregáveis definidos para o FT03.
